In [ ]:
%load_ext autoreload
%autoreload 2
# %matplotlib widget
%pdb off

from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, HTML
import pyafn
from pyafn import rho, Cd
import random
from scipy.stats import lognorm
from scipy.stats import norm
from scipy.optimize import curve_fit
from emulationHelpers import readEmulationMI, plot_ventilation_model_fit
from flowEmulationUtils import aggregate_window_series_to_room

#close all figures
plt.close('all')
plt.rcParams['figure.dpi'] = 140
im_scaling = .75
plt.rcParams['figure.figsize'] = [6.4 * im_scaling, 4.8 * im_scaling]
plt.rcParams['font.family'] = ['STIXGeneral', 'DejaVu Serif', 'serif']

home_dir = "./"
display(home_dir)
plt.close('all')

# Load Data

In [ ]:
flowStatsMI, roomVentilationMI = readEmulationMI(home_dir=home_dir)
NWindows = len(flowStatsMI[flowStatsMI['openingType'] == 'xwindow'])
NWindows += len(flowStatsMI[flowStatsMI['openingType'] == 'zwindow'])
display(f"N Windows: {NWindows}")
display(f"N Skylight: {len(flowStatsMI[flowStatsMI['openingType'] == 'skylight'])}")
display(f"N Rooms: {len(roomVentilationMI)}")

In [ ]:
y_var = "flux-Norm"
x_var = "p-noInt_optp0-q_model-Norm"
df = flowStatsMI.copy()
data = df.copy()
room_type_order = ["cross", "corner", "dual", "single"]
room_type_display_labels = {
    "cross": "Cross",
    "corner": "Corner",
    "dual": "Dual-Room",
    "single": "Single-Sided",
}
room_type_palette = dict(
    zip(room_type_order, sns.color_palette("colorblind", n_colors=len(room_type_order)))
)
model_line_label_order = [
    r"$\overline{q} = q_{MP}$",
    r"$\overline{q} = q_{MP}\sqrt{1-I^2}$",
    r"$\overline{q} = q_{MP}/(2H)$",
    r"$\overline{q} = q_{\mathrm{PW}}$",
]


In [ ]:
# Use the plotting function with turbulence model
fig, axs, metrics_ps_baseline = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="Mean-Pressure Baseline",
    p0=[1.0, 0.00000001],
    bounds=([0.99999, 0], [1.00001, 0.0000001]),
    adjustData=True,
    show_assymptotes=True,
    return_metrics=True,
)


In [ ]:
# Use the plotting function with turbulence model
fig, axs, metrics_ps_fit = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="Mean-Pressure Fit PN",
    p0=[1.0, 0.00000001],
    bounds=([0.1, 0], [np.inf, 0.0000001]),
    adjustData=False,
    show_assymptotes=True,
    return_metrics=True,
)


In [ ]:
# Use the plotting function with turbulence model
fig, axs, metrics_iq_fit = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ PN Bulk Fit",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True,
    return_metrics=True,
)


In [ ]:
x_var2 = "rms-mass_flux-Norm"
q_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2].mean().to_dict()
q_rms_global = data[x_var2].mean()
print(f"Global mean {x_var2}: {q_rms_global:.4f}")
for sl in [0, 1]:
    for s in [1, -1]:
        label = "Skylight" if sl == 1 else "Window"
        direction = "In" if s == 1 else "Out"
        print(f"{label}-{direction} mean {x_var2}: {q_rms_means.get((sl, s), np.nan):.4f}")

# Use subgroup-specific initialization for the second fit parameter
fig, axs, metrics_iq_bulk = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ PN Bulk Measured",
    p0=[1.0, q_rms_global],
    bounds=([0.1, q_rms_global-0.00001], [np.inf, q_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=q_rms_means,
    group_param2_half_width=0.00001,
    return_metrics=True,
 )


In [ ]:
def ventilationReDecomp_qMeasured(X, a, u_rms):
    u_model, rms_scaling = X
    u_rms *= rms_scaling
    return pyafn.ventilationReDecomp_q(u_model, a, u_rms)

plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=x_var2,
    hue=x_var2,
    style="slAll",
    model_func=ventilationReDecomp_qMeasured,
    model_name="Turbulence-Enhanced q",
    p0=[1.0, 1.0],
    bounds=([0.1, .9999], [np.inf, 1.00001]),
    adjustData=False,
    show_assymptotes=False,
    add_numeric_colorbar=True
 )

In [ ]:
# Use the plotting function with turbulence model
fig, axs, metrics_ip_fit = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="$\sigma_{\Delta C_p}$ PN Bulk Fit",
    p0=[1.0, 0.01],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True,
    return_metrics=True,
)


In [ ]:
x_var2 = "p_rms-noInt-Norm"
p_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2].mean().to_dict()
p_rms_global = data[x_var2].mean()
print(f"Global mean {x_var2}: {p_rms_global:.4f}")
for sl in [0, 1]:
    for s in [1, -1]:
        label = "Skylight" if sl == 1 else "Window"
        direction = "In" if s == 1 else "Out"
        print(f"{label}-{direction} mean {x_var2}: {p_rms_means.get((sl, s), np.nan):.4f}")

# Use subgroup-specific initialization for the second fit parameter
fig, axs, metrics_ip_bulk = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="$\sigma_{\Delta C_p}$ PN Bulk Measured",
    p0=[1.0, p_rms_global],
    bounds=([0.1, p_rms_global-0.00001], [np.inf, p_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=p_rms_means,
    group_param2_half_width=0.00001,
    return_metrics=True,
 )


In [ ]:
def ventilationReDecomp_pMeasured(X, a, p_rms):
    u_model, rms_scaling = X
    p_rms *= rms_scaling
    return pyafn.ventilationReDecomp_p(u_model, a, p_rms)

# Combined paper-ready q+p figure (rows: Window/Skylight; cols: q fit, q measured, p fit, p measured)
fig_p, axs_p = plt.subplots(2, 4, figsize=(12, 7), dpi=160, sharex=True, sharey=True)

# Column 1: baseline q model (categorical hue)
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ PN Bulk Fit",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True,
    axes=axs_p[:, 0],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Column 2: subgroup-mean initialized q model (categorical hue)
x_var2_q = "rms-mass_flux-Norm"
q_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2_q].mean().to_dict()
q_rms_global = data[x_var2_q].mean()
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ PN Bulk Measured",
    p0=[1.0, q_rms_global],
    bounds=([0.1, q_rms_global-0.00001], [np.inf, q_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=q_rms_means,
    group_param2_half_width=0.00001,
    axes=axs_p[:, 1],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Column 3: baseline p model (categorical hue)
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="$\sigma_{\Delta C_p}$ PN Bulk Fit",
    p0=[1.0, 0.01],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True,
    axes=axs_p[:, 2],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Column 4: subgroup-mean initialized p model (categorical hue)
x_var2_p = "p_rms-noInt-Norm"
p_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2_p].mean().to_dict()
p_rms_global = data[x_var2_p].mean()
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="$\sigma_{\Delta C_p}$ PN Bulk Measured",
    p0=[1.0, p_rms_global],
    bounds=([0.1, p_rms_global-0.00001], [np.inf, p_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=p_rms_means,
    group_param2_half_width=0.00001,
    axes=axs_p[:, 3],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Paper-ready labels and layout
model_labels_p = [
    "$\sigma_{q_n}$ PN Bulk Fit",
    "$\sigma_{q_n}$ PN Bulk Measured",
    "$\sigma_{\Delta C_p}$ PN Bulk Fit",
    "$\sigma_{\Delta C_p}$ PN Bulk Measured",
]

for ax in axs_p.flatten():
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.tick_params(labelsize=16)

for j, model_label in enumerate(model_labels_p):
    axs_p[0, j].set_title(model_label, fontsize=20, pad=10)

row_labels_p = ["Windows", "Skylights"]
for i, row_label in enumerate(row_labels_p):
    axs_p[i, 0].set_ylabel(row_label, fontsize=20, labelpad=12)

line_handles, line_labels = axs_p[0, 0].get_legend_handles_labels()
line_lookup = dict(zip(line_labels, line_handles))
model_legend_items = [(line_lookup[label], label) for label in model_line_label_order if label in line_lookup]
if model_legend_items:
    model_legend = fig_p.legend(
        [handle for handle, _ in model_legend_items],
        [label for _, label in model_legend_items],
        loc="lower center",
        bbox_to_anchor=(0.5, 0.09),
        ncol=len(model_legend_items),
        frameon=False,
        fontsize=12,
    )

room_handles_p = [
    Line2D(
        [0], [0],
        color=room_type_palette[room_type],
        marker="o",
        linestyle="None",
        markersize=7,
        label=room_type_display_labels[room_type],
    )
    for room_type in room_type_order
]
fig_p.legend(
    room_handles_p,
    [handle.get_label() for handle in room_handles_p],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.03),
    ncol=len(room_handles_p),
    frameon=False,
    fontsize=12,
)
fig_p.supxlabel("Baseline $q_{MP,n}$", fontsize=24, y=0.17, x = 0.54)
fig_p.supylabel("LES $\overline{q_n}$", fontsize=24, x=-0.01, y=0.6)
# fig_p.suptitle("Ventilation Model Fits (q and p): Window vs Skylight", fontsize=24)
fig_p.subplots_adjust(left=0.10, right=0.98, top=0.94, bottom=0.28, wspace=0.16, hspace=0.18)


In [ ]:
# Combined measured-only q+p figure (rows: Window/Skylight; cols: q measured, p measured)
fig_m, axs_m = plt.subplots(2, 2, figsize=(9.6, 8), dpi=160, sharex=True, sharey=True, layout="constrained")

# Column 1: subgroup-mean initialized q model (categorical hue)
_, _, params_m_q = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style=None,
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ PN Bulk Measured",
    p0=[1.0, q_rms_global],
    bounds=([0.1, q_rms_global-0.00001], [np.inf, q_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=q_rms_means,
    group_param2_half_width=0.00001,
    axes=axs_m[:, 0],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
    marker_size=7.5,
    return_params=True,
 )

# Column 2: subgroup-mean initialized p model (categorical hue)
_, _, params_m_p = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style=None,
    model_func=pyafn.ventilationReDecomp_p,
    model_name="$\sigma_{\Delta C_p}$ PN Bulk Measured",
    p0=[1.0, p_rms_global],
    bounds=([0.1, p_rms_global-0.00001], [np.inf, p_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=p_rms_means,
    group_param2_half_width=0.00001,
    axes=axs_m[:, 1],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
    marker_size=7.5,
    return_params=True,
 )

# Paper-ready labels and layout
model_labels_m = [
    "$\sigma_{q_n}$",
    "$\sigma_{\Delta C_p}$",
]

for ax in axs_m.flatten():
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.tick_params(labelsize=16)

for j, model_label in enumerate(model_labels_m):
    axs_m[0, j].set_title(model_label, fontsize=20, pad=10)

row_labels_m = ["Windows", "Skylights"]
for i, row_label in enumerate(row_labels_m):
    axs_m[i, 0].set_ylabel(row_label, fontsize=20, labelpad=-6)

from matplotlib.colors import same_color

# Same axis limits as the opening-specific IN scatterplots
for ax in axs_m.flatten():
    ax.set_xlim(-0.6, 0.6)
    ax.set_ylim(-0.6, 0.6)
    # Restyle the plot_ventilation_model_fit lines: the blue and green asymptotes take the
    # fit's width, and the black q_PW fit drops to the width of the red 1:1 line and is drawn on top
    model_lines = ax.get_lines()
    fit_width = next((line.get_linewidth() for line in model_lines if same_color(line.get_color(), "k")), 2.5)
    reference_width = next((line.get_linewidth() for line in model_lines if same_color(line.get_color(), "r")), 1.5)
    for line in model_lines:
        if same_color(line.get_color(), "green") or same_color(line.get_color(), "blue"):
            line.set_linewidth(fit_width)
        elif same_color(line.get_color(), "k"):
            line.set_linewidth(reference_width)
            line.set_zorder(3)  # draw the fit over the other lines (default zorder 2)

# I* lines: baseline q_MP,n where each subgroup fit switches between the mean- and fluctuation-dominated corrections
#   sigma_qn:  I = sigma / (a q_MP)                       -> q_MP* = sigma / (a I*),                    I* = 1/sqrt(2)
#   sigma_dCp: I = sigma / (2 (q_MP / (C_d sqrt(2/rho)))^2) -> q_MP* = C_d sqrt(2/rho) sqrt(sigma / (2 I*)), I* = 1/sqrt(3)
from pyafn.constants import Cd as pyafn_Cd, rho as pyafn_rho

I_star_specs = [
    (params_m_q, 1 / np.sqrt(2), lambda a, sigma, I_star: sigma / (a * I_star)),
    (params_m_p, 1 / np.sqrt(3), lambda a, sigma, I_star: pyafn_Cd * np.sqrt(2 / pyafn_rho) * np.sqrt(sigma / (2 * I_star))),
]
for col_idx, (params, I_star, x_star_from_params) in enumerate(I_star_specs):
    for row_idx, opening_group in enumerate(["Window", "Skylight"]):
        ax = axs_m[row_idx, col_idx]
        for direction, sign in [("Flow Entering", 1), ("Flow Exiting", -1)]:
            if (opening_group, direction) not in params:
                continue
            a, sigma = params[(opening_group, direction)][:2]
            x_star = sign * x_star_from_params(a, sigma, I_star)
            if not ax.get_xlim()[0] < x_star < ax.get_xlim()[1]:
                continue
            ax.axvline(x_star, color="0.3", linestyle="-.", linewidth=1, zorder=1)
            # Label each line on its right in the first panel only: the outflow line at the top (left of it
            # is the "Flow in" R^2 label), the inflow line centered on y = 0 so the two don't overlap
            if (row_idx, col_idx) == (0, 0):
                ax.annotate(
                    "$q_{MP,n}(I^*)$", (x_star, 0) if sign > 0 else (x_star, 0.88),
                    xycoords="data" if sign > 0 else ax.get_xaxis_transform(),
                    xytext=(3, 0), textcoords="offset points", ha="left",
                    va="center" if sign > 0 else "top", fontsize=14, color="0.3",
                )

# Model-line legend in the bottom-right of the top-left panel, just above its "Flow out" R^2 label
legend_kwargs = dict(
    fontsize=12, frameon=True, facecolor="white", edgecolor="0.6", framealpha=0.9, fancybox=False,
    handletextpad=0.3, borderpad=0.3, labelspacing=0.3,
)
lines_by_label = {line.get_label(): line for line in axs_m[0, 0].get_lines()}
line_legend_items = [(lines_by_label[label], label) for label in model_line_label_order if label in lines_by_label]
# Show the normalized forms of the model lines
line_legend_display = {
    r"$\overline{q} = q_{MP}$": r"$\overline{q}_n = q_{MP,n}$",
    r"$\overline{q} = q_{MP}\sqrt{1-I^2}$": r"$\overline{q}_n = q_{MP,n}\sqrt{1-I^2}$",
    r"$\overline{q} = q_{MP}/(2H)$": r"$\overline{q}_n = q_{MP,n}/(2H_U)$",
    r"$\overline{q} = q_{\mathrm{PW}}$": r"$\overline{q}_n = q_{\mathrm{PW},n}$",
}
line_legend = axs_m[0, 0].legend(
    [handle for handle, _ in line_legend_items],
    [line_legend_display.get(label, label) for _, label in line_legend_items],
    loc="lower right",
    bbox_to_anchor=(1.0, 0.08),
    **legend_kwargs,
)
axs_m[0, 0].add_artist(line_legend)  # keep it when the room-type legend below replaces the axes legend

# Room-type legend in the top-left panel, just below its "Flow in" R^2 label
axs_m[0, 0].legend(
    room_handles_p,
    [handle.get_label() for handle in room_handles_p],
    loc="upper left",
    bbox_to_anchor=(0.0, 0.92),
    **legend_kwargs,
)
# Axis labels placed like the opening-specific IN figures (same figure size, so the same offsets)
fig_m.supxlabel("Baseline $q_{MP,n}$", fontsize=24, y=-0.05, x=0.53)
fig_m.supylabel("LES $\overline{q_n}$", fontsize=24, x=-0.036, y=0.51)

In [ ]:
data["Cp"] = data["p_rms-noInt-Norm"] / (rho / 2)

# Combined opening-specific figure (rows: Window/Skylight; cols: q and p)
fig_pointwise, axs_pointwise = plt.subplots(2, 2, figsize=(12, 10), dpi=160, sharex=True, sharey=True, layout='constrained')

# Opening-specific q (left column)
q_hue_norm = plt.Normalize(vmin=0, vmax=0.25)
data_q_pointwise = data.copy()
_, _, metrics_iq_pointwise = plot_ventilation_model_fit(
    data=data_q_pointwise,
    y_var=y_var,
    x_var=x_var,
    x_var2=x_var2_q,
    hue=x_var2_q,
    style=None,
    model_func=ventilationReDecomp_qMeasured,
    model_name="$\sigma_{q_n}$ PN Opening-Specific Measured",
    p0=[1.0, 1.0],
    bounds=([0.1, .9999], [np.inf, 1.00001]),
    adjustData=True,
    show_assymptotes=True,
    axes=axs_pointwise[:, 0],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=True,
    hue_norm=q_hue_norm,
    palette_numeric="plasma",
    figure_suptitle=False,
    colorbar_rect=[0.16, -0.1, 0.30, 0.018],
    colorbar_orientation="horizontal",
    colorbar_label=r"$\sigma_{q_n}$",
    return_metrics=True,
)

# Opening-specific p (right column)
x_var2_p = "p_rms-noInt-Norm"
p_hue_norm = plt.Normalize(vmin=0, vmax=0.25)

data_p_pointwise = data.copy()
_, _, metrics_ip_pointwise = plot_ventilation_model_fit(
    data=data_p_pointwise,
    y_var=y_var,
    x_var=x_var,
    x_var2=x_var2_p,
    hue="Cp",
    style=None,
    model_func=ventilationReDecomp_pMeasured,
    model_name="$\sigma_{\Delta C_p}$ PN Opening-Specific Measured",
    p0=[1.0, 1.0],
    bounds=([0.1, .9999], [np.inf, 1.00001]),
    adjustData=True,
    show_assymptotes=False,
    axes=axs_pointwise[:, 1],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=True,
    hue_norm=p_hue_norm,
    palette_numeric="viridis",
    figure_suptitle=False,
    colorbar_rect=[0.61, -0.1, 0.30, 0.018],
    colorbar_orientation="horizontal",
    colorbar_label=r"$\sigma_{\Delta C_p}$",
    return_metrics=True,
)

# Labels and layout
model_labels_pointwise = [
    "$\sigma_{q_n}$",
    "$\sigma_{\Delta C_p}$",
]

for ax in axs_pointwise.flatten():
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.tick_params(labelsize=16)

for j, model_label in enumerate(model_labels_pointwise):
    axs_pointwise[0, j].set_title(model_label, fontsize=18, pad=10)

row_labels_pointwise = ["Windows", "Skylights"]
for i, row_label in enumerate(row_labels_pointwise):
    axs_pointwise[i, 0].set_ylabel(row_label, fontsize=18, labelpad=12)

fig_pointwise.supxlabel("$q_{PW,n}$", fontsize=24, y=-0.05, x=0.53)
fig_pointwise.supylabel("LES $\overline{q_n}$", fontsize=24, x=-0.01, y=0.51)
# fig_pointwise.suptitle("Opening-Specific Ventilation Model Fits", fontsize=24)
fig_pointwise.subplots_adjust(left=0.12, right=0.94, top=0.90, bottom=0.2, wspace=0.2, hspace=0.22)



In [ ]:
# Combined opening-specific figure colored by fluctuation correction (q_PW = a * q_MP * correction)
def add_fluctuation_correction(df, metrics, model_func, x_var2, col):
    """Add the blended fluctuation scaling for each point using the subgroup-fitted parameters."""
    df = df.copy()
    df[col] = np.nan
    for _, row in metrics[metrics["scope"] == "subgroup"].iterrows():
        sl_val = row["opening_group"] == "Skylight"
        s = 1 if row["direction"] == "Flow Entering" else -1
        mask = (df["skylight"] == sl_val) & (df["Sdelp"] == s)
        a = row["cd"] / Cd
        x = np.abs(df.loc[mask, x_var].to_numpy())
        x2 = np.abs(df.loc[mask, x_var2].to_numpy())
        with np.errstate(divide="ignore", invalid="ignore"):
            df.loc[mask, col] = model_func((x, x2), a, row["sigma"]) / (a * x)
    return df

data_corr = add_fluctuation_correction(data, metrics_iq_pointwise, ventilationReDecomp_qMeasured, x_var2_q, "corr_q")
data_corr = add_fluctuation_correction(data_corr, metrics_ip_pointwise, ventilationReDecomp_pMeasured, x_var2_p, "corr_p")
print(data_corr[["corr_q", "corr_p"]].describe())

corr_hue_norm = plt.Normalize(vmin=0, vmax=1)

fig_corr, axs_corr = plt.subplots(2, 2, figsize=(12, 10), dpi=160, sharex=True, sharey=True, layout='constrained')

# Opening-specific q (left column)
plot_ventilation_model_fit(
    data=data_corr,
    y_var=y_var,
    x_var=x_var,
    x_var2=x_var2_q,
    hue="corr_q",
    style=None,
    model_func=ventilationReDecomp_qMeasured,
    model_name="$\sigma_{q_n}$ PN Opening-Specific Measured",
    p0=[1.0, 1.0],
    bounds=([0.1, .9999], [np.inf, 1.00001]),
    adjustData=True,
    show_assymptotes=True,
    axes=axs_corr[:, 0],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=True,
    hue_norm=corr_hue_norm,
    palette_numeric="viridis",
    figure_suptitle=False,
    colorbar_rect=[1.01, 0.08, 0.02, 0.84],
    colorbar_orientation="vertical",
    colorbar_label=r"Fluctuation correction $q_{PW,n} / q_{MP,n}$",
)

# Opening-specific p (right column), sharing the colorbar above
plot_ventilation_model_fit(
    data=data_corr,
    y_var=y_var,
    x_var=x_var,
    x_var2=x_var2_p,
    hue="corr_p",
    style=None,
    model_func=ventilationReDecomp_pMeasured,
    model_name="$\sigma_{\Delta C_p}$ PN Opening-Specific Measured",
    p0=[1.0, 1.0],
    bounds=([0.1, .9999], [np.inf, 1.00001]),
    adjustData=True,
    show_assymptotes=False,
    axes=axs_corr[:, 1],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    hue_norm=corr_hue_norm,
    palette_numeric="viridis",
    figure_suptitle=False,
)

# Labels and layout
for ax in axs_corr.flatten():
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.tick_params(labelsize=16)

for j, model_label in enumerate(model_labels_pointwise):
    axs_corr[0, j].set_title(model_label, fontsize=18, pad=10)

for i, row_label in enumerate(row_labels_pointwise):
    axs_corr[i, 0].set_ylabel(row_label, fontsize=18, labelpad=12)

fig_corr.supxlabel("$q_{PW,n}$", fontsize=24, y=-0.05, x=0.53)
fig_corr.supylabel("LES $\overline{q_n}$", fontsize=24, x=-0.01, y=0.51)
fig_corr.subplots_adjust(left=0.12, right=0.94, top=0.90, bottom=0.2, wspace=0.2, hspace=0.22)


In [ ]:
# Row 1: baseline q model (categorical hue)
fig, ax, xAdjusted, fittedParams = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ PN Bulk Fit",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=True,
    show_assymptotes=True,
    add_numeric_colorbar=False,
    return_data=True,
    return_params=True
)

In [ ]:
plt.figure()
plt.scatter(xAdjusted, data[y_var], alpha=0.5, s=1)

In [ ]:
minVent = roomVentilationMI[x_var].min()
maxVent = roomVentilationMI[x_var].max()
x_space = np.linspace(minVent, maxVent, 100)

windowModels = {}
for key, params in fittedParams.items():
    windowModels[key] = pyafn.ventilationReDecomp_q(x_space, *params)

roomModels = {}
roomResiduals = {}
for inOpening in ["Window", "Skylight"]:
    for outOpening in ["Window", "Skylight"]:
        roomModels[f"In {inOpening}; Out {outOpening}"] = (windowModels[(inOpening, 'Flow Entering')] + windowModels[(outOpening, 'Flow Exiting')]) / 2
        roomResiduals[f"In {inOpening}; Out {outOpening}"] = windowModels[(inOpening, 'Flow Entering')] - windowModels[(outOpening, 'Flow Exiting')]


In [ ]:
roomVentilationMI_adj = aggregate_window_series_to_room(
    roomVentilationMI, xAdjusted, out_col="xAdjusted_room", mode="abs_half_sum"
)

roomVentilationMI_adj = aggregate_window_series_to_room(
    roomVentilationMI_adj, xAdjusted, out_col="xAdjusted_room_residual", mode="mean"
)
roomVentilationMI_adj["xAdjusted_room_residual"] *= 2
roomVentilationMI_adj["xAdjusted_diff"] = roomVentilationMI_adj["xAdjusted_room"] - roomVentilationMI_adj[x_var]

plot_df = roomVentilationMI_adj.copy().sort_values("roomType")

fig, axs = plt.subplots(2, 2, figsize=(13, 11), dpi=160)

for i in range(2):
    ax = axs[0, i]
    ax.plot(x_space, x_space, 'r--', label='1:1 Line')

styles = ['-', '--', '-.', ':']
for i, (key, model) in enumerate(roomModels.items()):
    color = "k"#plt.get_cmap("tab20")(i % 20)
    axs[1, 0].plot(x_space, model, linestyle=styles[i % len(styles)], label=key, color=color, alpha=1, linewidth = 1)

for i, (key, residual) in enumerate(roomResiduals.items()):
    color = "k" #plt.get_cmap("tab20")(i % 20)
    axs[1, 1].plot(
        roomModels[key] - x_space,
        residual,
        linestyle=styles[i % len(styles)],
        label=key,
        color=color,
        alpha=1,
        linewidth = 1
    )
    axs[1, 1].set_xlim(-0.1, 0.1)
    axs[1, 1].set_ylim(-0.1, 0.1)

scatter_specs = [
    (x_var, "flux-Norm", "Baseline Model vs LES"),
    ("xAdjusted_room", "flux-Norm", "Aggregated Adjusted Model vs LES"),
    (x_var, "xAdjusted_room", "Baseline Model vs Aggregated Adjusted Model"),
    ("xAdjusted_diff", "xAdjusted_room_residual", "Aggregation Difference vs Residual"),
]

for ax, (xcol, ycol, title) in zip(axs.flatten(), scatter_specs):
    sns.scatterplot(
        data=plot_df,
        x=xcol,
        y=ycol,
        hue="roomType",
        style = "slAll",
        palette="colorblind",
        hue_order=room_type_order,
        alpha=0.6,
        s=10,
        ax=ax,
        legend=ax is axs[0, 0],
    )
 
    ax.set_title(title, fontsize=20)
    ax.set_xlabel(xcol, fontsize=18)
    ax.set_ylabel(ycol, fontsize=18)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=16)

room_handles = [
    Line2D(
        [0], [0],
        color=room_type_palette[room_type],
        marker="o",
        linestyle="None",
        markersize=8,
        label=room_type_display_labels[room_type],
    )
    for room_type in room_type_order
]

for ax in axs.flatten():
    if ax.get_legend() is not None:
        ax.get_legend().remove()

axs[1, 0].legend(fontsize=10, title="Opening Pair")
axs[1, 1].legend(fontsize=10, title="Opening Pair")

fig.legend(
    room_handles,
    [handle.get_label() for handle in room_handles],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=len(room_handles),
    fontsize=14,
    frameon=False,
)

fig.suptitle("Room-Level Aggregated Ventilation Comparisons", fontsize=24)
fig.subplots_adjust(left=0.10, right=0.97, top=0.90, bottom=0.18, wspace=0.28, hspace=0.30)

In [ ]:
def _params_for_opening(opening_type):
    group = "Skylight" if "skylight" in str(opening_type).lower() else "Window"
    return (
        [fittedParams[(group, "Flow Entering")][0]*Cd, fittedParams[(group, "Flow Exiting")][0]*Cd],
        [fittedParams[(group, "Flow Entering")][1], fittedParams[(group, "Flow Exiting")][1]],
    )

def _direct_network_param_lookup_from_fitted_params(fitted_params, measured_sigma=False):
    lookup = {}
    for (opening_group, direction), params in fitted_params.items():
        lookup[(opening_group, direction)] = {
            "cd": params[0] * Cd,
            "sigma": np.nan if measured_sigma else (params[1] if len(params) > 1 else np.nan),
        }
    return lookup

flowStatsMI[["C_d", "pRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

direct_network_param_lookup = _direct_network_param_lookup_from_fitted_params(fittedParams)

flowStatsMI[["C_d", "pRMS"]]

In [ ]:
import flowEmulationUtils as feUtils
flowStatsMI_directNetwork, roomVentilationMI_directNetwork = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [Cd, Cd], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
def _comparison_metrics(frame, x_spec, y_col):
    if isinstance(x_spec, str):
        x = frame[x_spec]
        x_label = x_spec
    else:
        x = pd.Series(x_spec).reindex(frame.index)
        x_label = getattr(x_spec, "name", None) or "x_adjusted"

    valid = pd.DataFrame({"x": x, "y": frame[y_col]}).dropna()
    residual = valid["x"] - valid["y"]
    rmse = np.sqrt(np.mean(residual ** 2))
    y_std = np.std(valid["y"])
    r_squared = 1 - np.sum(residual ** 2) / np.sum((valid["y"] - valid["y"].mean()) ** 2) if y_std > 0 else np.nan
    iqr_error = residual.quantile(0.75) - residual.quantile(0.25)

    return {
        "x_label": x_label,
        "rmse": rmse,
        "nrmse": rmse / y_std if y_std > 0 else np.nan,
        "r_squared": r_squared,
        "bias": residual.mean(),
        "iqr_error": iqr_error,
        "std": residual.std(ddof=0),
        "n": len(valid),
    }


def _comparison_row(frame, x_spec, y_col, panel, split):
    metrics = _comparison_metrics(frame, x_spec, y_col)
    if metrics["n"] == 0:
        return None

    return {
        "panel": panel,
        "split": split,
        "rmse": metrics["rmse"],
        "nrmse": metrics["nrmse"],
        "r_squared": metrics["r_squared"],
        "bias": metrics["bias"],
        "iqr_error": metrics["iqr_error"],
        "std": metrics["std"],
        "n": metrics["n"],
        "x_label": metrics["x_label"],
    }


def _annotate_directional_r_squared(ax, frame, x_spec, y_col, fontsize=12):
    if "Sdelp" not in frame.columns:
        return

    annotation_specs = [
        (frame["Sdelp"] > 0, "Flow in", (0.03, 0.97), "left", "top", "tab:red"),
        (frame["Sdelp"] < 0, "Flow out", (0.97, 0.03), "right", "bottom", "tab:blue"),
    ]
    for mask, label, position, horizontal_alignment, vertical_alignment, color in annotation_specs:
        metrics = _comparison_metrics(frame.loc[mask], x_spec, y_col)
        if metrics["n"] == 0:
            continue
        ax.text(
            *position,
            rf"{label}: $R^2={metrics['r_squared']:.2f}$",
            transform=ax.transAxes,
            ha=horizontal_alignment,
            va=vertical_alignment,
            fontsize=fontsize,
            color=color,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.75, "pad": 1.5},
            zorder=5,
        )


def _flow_subgroup_masks(frame):
    if "Sdelp" not in frame.columns:
        return []

    if "openingType" in frame.columns:
        opening_group = pd.Series(
            np.where(
                frame["openingType"].astype(str).str.contains("skylight", case=False, na=False),
                "Skylight",
                "Window",
            ),
            index=frame.index,
        )
    elif "skylight" in frame.columns:
        opening_group = pd.Series(
            np.where(frame["skylight"].fillna(0).astype(int) == 1, "Skylight", "Window"),
            index=frame.index,
        )
    else:
        return []

    direction_group = pd.Series(None, index=frame.index, dtype="object")
    direction_group.loc[frame["Sdelp"] > 0] = "Flow Entering"
    direction_group.loc[frame["Sdelp"] < 0] = "Flow Exiting"

    masks = []
    for opening_label in ["Window", "Skylight"]:
        for direction_label in ["Flow Entering", "Flow Exiting"]:
            mask = (opening_group == opening_label) & (direction_group == direction_label)
            if mask.any():
                masks.append((f"{opening_label}, {direction_label}", mask))

    return masks


def _panel_plot_data(frame, x_spec):
    plot_frame = frame.copy()
    if isinstance(x_spec, str):
        return plot_frame, x_spec

    x_label = getattr(x_spec, "name", None) or "x_adjusted"
    plot_frame[x_label] = pd.Series(x_spec).reindex(plot_frame.index)
    return plot_frame, x_label


def plot_direct_network_comparison(run_label, flow_frame, room_frame):
    flow_plot = flow_frame.copy()
    room_plot = room_frame.copy()

    flow_plot["p-noInt-direct_optp0-q_model-Norm"] = (
        flow_plot["p-noInt-direct_optp0-q_model"] / flow_plot["WS"]
    )
    room_plot["p-noInt-direct_optp0-q_model-Norm"] = (
        room_plot["p-noInt-direct_optp0-q_model"] / room_plot["WS"]
    )
    flow_plot["flux-Norm"] = flow_plot["flux"] / flow_plot["WS"]
    for measured_col in ["p_rms-noInt-Norm", "rms-mass_flux-Norm"]:
        if measured_col not in flow_plot.columns and measured_col in data.columns:
            flow_plot[measured_col] = data[measured_col].reindex(flow_plot.index)
    if "Cp" not in flow_plot.columns and "p_rms-noInt-Norm" in flow_plot.columns:
        flow_plot["Cp"] = flow_plot["p_rms-noInt-Norm"] / (rho / 2)

    hue_order = room_type_order
    slall_style_col = "Skylight State"
    slall_style_order = ["Interiors Without Skylights", "Interiors With Skylights"]
    measured_opening_hue = {
        "$\sigma_{\Delta C_p}$ IN Opening-Specific Measured": {
            "hue": "Cp",
            "palette": "viridis",
            "norm": plt.Normalize(vmin=0, vmax=0.25),
            "label": r"$\sigma_{\Delta C_p}$",
        },
        "$\sigma_{q_n}$ IN Opening-Specific Measured": {
            "hue": "rms-mass_flux-Norm",
            "palette": "plasma",
            "norm": plt.Normalize(vmin=0, vmax=0.25),
            "label": r"$\sigma_{q_n}$",
        },
    }.get(run_label)
    pn_model_label = run_label.replace(" IN ", " PN ", 1) if " IN " in run_label else "PN Model"
    in_model_label = run_label if " IN " in run_label else "IN Model"
    column_labels = ["Mean-Pressure Baseline", pn_model_label, in_model_label]
    row_labels = ["Windows", "Skylights", "Interiors Without Skylights", "Interiors With Skylights"]

    def _skylight_mask(frame):
        if "openingType" in frame.columns:
            return frame["openingType"].astype(str).str.contains("skylight", case=False, na=False)
        return frame["skylight"].fillna(0).astype(int) == 1

    def _opening_hue_spec(frame):
        if measured_opening_hue is None or measured_opening_hue["hue"] not in frame.columns:
            return {"hue": "roomType", "palette": "colorblind", "hue_order": hue_order}
        return {
            "hue": measured_opening_hue["hue"],
            "palette": measured_opening_hue["palette"],
            "hue_norm": measured_opening_hue["norm"],
            "hue_order": None,
            "numeric_hue": True,
        }

    def _add_slall_style(frame):
        frame = frame.copy()
        if "slAll" in frame.columns:
            frame[slall_style_col] = np.where(frame["slAll"].astype(bool), "Interiors With Skylights", "Interiors Without Skylights")
        else:
            frame[slall_style_col] = "Unknown"
        return frame

    window_data = _add_slall_style(data[~_skylight_mask(data)])
    skylight_data = _add_slall_style(data[_skylight_mask(data)])
    window_flow_plot = _add_slall_style(flow_plot[~_skylight_mask(flow_plot)])
    skylight_flow_plot = _add_slall_style(flow_plot[_skylight_mask(flow_plot)])

    non_slall_room_base = roomVentilationMI[roomVentilationMI["slAll"] == False]
    slall_room_base = roomVentilationMI[roomVentilationMI["slAll"] == True]
    non_slall_room_adj = roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == False]
    slall_room_adj = roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == True]
    non_slall_room_plot = room_plot[room_plot["slAll"] == False]
    slall_room_plot = room_plot[room_plot["slAll"] == True]

    panel_grid = [
        [
            {
                "title": "Window Baseline vs LES",
                "frame": window_data,
                "x": x_var,
                "y": y_var,
                **_opening_hue_spec(window_data),
                "style": slall_style_col,
                "style_order": slall_style_order,
            },
            {
                "title": "Window Adjusted vs LES",
                "frame": window_flow_plot,
                "x": xAdjusted,
                "y": y_var,
                **_opening_hue_spec(window_flow_plot),
                "style": slall_style_col,
                "style_order": slall_style_order,
            },
            {
                "title": f"Window {run_label} vs LES",
                "frame": window_flow_plot,
                "x": "p-noInt-direct_optp0-q_model-Norm",
                "y": y_var,
                **_opening_hue_spec(window_flow_plot),
                "style": slall_style_col,
                "style_order": slall_style_order,
            },
        ],
        [
            {
                "title": "Skylight Baseline vs LES",
                "frame": skylight_data,
                "x": x_var,
                "y": y_var,
                **_opening_hue_spec(skylight_data),
                "style": slall_style_col,
                "style_order": slall_style_order,
            },
            {
                "title": "Skylight Adjusted vs LES",
                "frame": skylight_flow_plot,
                "x": xAdjusted,
                "y": y_var,
                **_opening_hue_spec(skylight_flow_plot),
                "style": slall_style_col,
                "style_order": slall_style_order,
            },
            {
                "title": f"Skylight {run_label} vs LES",
                "frame": skylight_flow_plot,
                "x": "p-noInt-direct_optp0-q_model-Norm",
                "y": y_var,
                **_opening_hue_spec(skylight_flow_plot),
                "style": slall_style_col,
                "style_order": slall_style_order,
            },
        ],
        [
            {
                "title": "Non-slAll Room Baseline vs LES",
                "frame": non_slall_room_base,
                "x": x_var,
                "y": y_var,
                "hue": "roomType",
                "palette": "colorblind",
                "hue_order": hue_order,
                "style": "roomType",
                "style_order": hue_order,
            },
            {
                "title": "Non-slAll Room Adjusted vs LES",
                "frame": non_slall_room_adj,
                "x": "xAdjusted_room",
                "y": y_var,
                "hue": "roomType",
                "palette": "colorblind",
                "hue_order": hue_order,
                "style": "roomType",
                "style_order": hue_order,
            },
            {
                "title": f"Non-slAll Room {run_label} vs LES",
                "frame": non_slall_room_plot,
                "x": "p-noInt-direct_optp0-q_model-Norm",
                "y": y_var,
                "hue": "roomType",
                "palette": "colorblind",
                "hue_order": hue_order,
                "style": "roomType",
                "style_order": hue_order,
            },
        ],
        [
            {
                "title": "slAll Room Baseline vs LES",
                "frame": slall_room_base,
                "x": x_var,
                "y": y_var,
                "hue": "roomType",
                "palette": "colorblind",
                "hue_order": hue_order,
                "style": "roomType",
                "style_order": hue_order,
            },
            {
                "title": "slAll Room Adjusted vs LES",
                "frame": slall_room_adj,
                "x": "xAdjusted_room",
                "y": y_var,
                "hue": "roomType",
                "palette": "colorblind",
                "hue_order": hue_order,
                "style": "roomType",
                "style_order": hue_order,
            },
            {
                "title": f"slAll Room {run_label} vs LES",
                "frame": slall_room_plot,
                "x": "p-noInt-direct_optp0-q_model-Norm",
                "y": y_var,
                "hue": "roomType",
                "palette": "colorblind",
                "hue_order": hue_order,
                "style": "roomType",
                "style_order": hue_order,
            },
        ],
    ]

    fig, axs = plt.subplots(4, 3, figsize=(11, 12), dpi=160, layout="constrained")
    stats_rows = []
    legend_ax = axs[2, 0] if measured_opening_hue is not None else axs[0, 0]

    flat_specs = [spec for row in panel_grid for spec in row]
    for panel_idx, (ax, spec) in enumerate(zip(axs.flatten(), flat_specs)):
        row_idx, col_idx = divmod(panel_idx, len(column_labels))
        plot_frame, x_col = _panel_plot_data(spec["frame"], spec["x"])
        scatter_kwargs = {
            "data": plot_frame,
            "x": x_col,
            "y": spec["y"],
            "hue": spec["hue"],
            "style": spec["style"],
            "palette": spec["palette"],
            "ax": ax,
            "alpha": 0.7,
            "s": 20,
            "hue_order": spec.get("hue_order"),
            "style_order": spec["style_order"],
            "legend": False if spec.get("numeric_hue", False) else ax is legend_ax,
        }
        if "hue_norm" in spec:
            scatter_kwargs["hue_norm"] = spec["hue_norm"]
        sns.scatterplot(**scatter_kwargs)

        metrics = _comparison_metrics(spec["frame"], spec["x"], spec["y"])
        total_row = _comparison_row(spec["frame"], spec["x"], spec["y"], spec["title"], "Total")
        if total_row is not None:
            stats_rows.append(total_row)

        for split_label, mask in _flow_subgroup_masks(spec["frame"]):
            split_row = _comparison_row(
                spec["frame"].loc[mask],
                spec["x"],
                spec["y"],
                spec["title"],
                split_label,
            )
            if split_row is not None:
                stats_rows.append(split_row)

        if row_idx < 2:
            _annotate_directional_r_squared(ax, spec["frame"], spec["x"], spec["y"], fontsize=11)

        valid = plot_frame[[x_col, spec["y"]]].dropna()
        if not valid.empty:
            lim_min = valid.min().min()
            lim_max = valid.max().max()
            ax.plot([lim_min, lim_max], [lim_min, lim_max], "r--", linewidth=1.0, alpha=0.7)
            ax.set_xlim(lim_min, lim_max)
            ax.set_ylim(lim_min, lim_max)

        ax.set_title(column_labels[col_idx] if row_idx == 0 else "", fontsize=16, pad=10)
        ax.set_xlabel("")
        ax.set_ylabel(row_labels[row_idx] if col_idx == 0 else "", fontsize=16, labelpad=10)
        ax.tick_params(axis="both", labelsize=14)
        ax.grid(True, alpha=0.3)

    handles, labels = axs[2, 0].get_legend_handles_labels()
    if not handles:
        handles, labels = axs[0, 0].get_legend_handles_labels()
    for ax in axs.flatten():
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    legend_y = 0.5
    legend_x = 0.98
    xlabel_x = 0.56

    if measured_opening_hue is not None:
        mappable = plt.cm.ScalarMappable(
            norm=measured_opening_hue["norm"],
            cmap=plt.get_cmap(measured_opening_hue["palette"]),
        )
        mappable.set_array([])
        cbar = fig.colorbar(mappable, ax=axs[:2, :], location="right", fraction=0.025, pad=0.02)
        cbar.set_label(measured_opening_hue["label"], fontsize=16)
        cbar.ax.tick_params(labelsize=14)
        legend_y = 0.25
        legend_x = 0.88
        xlabel_x = 0.50

    if handles:
        fig.legend(
            handles,
            labels,
            loc="center left",
            bbox_to_anchor=(legend_x, legend_y),
            fontsize=14,
            frameon=False,
        )



    fig.supxlabel("$q_{PW,n}$", fontsize=18, y=-0.03, x=xlabel_x)
    fig.supylabel("LES $\overline{q_n}$", fontsize=18, x=-0.01, y=0.49)
    fig.suptitle(f"{run_label} Comparison", fontsize=18)
    fig.subplots_adjust(left=0.10, right=0.88, bottom=0.08, hspace=0.45, wspace=0.25)

    stats_df = pd.DataFrame(stats_rows)
    print(run_label)
    overall_stats = stats_df[stats_df["split"] == "Total"].drop(columns=["split", "x_label"])
    print("Overall")
    print(overall_stats.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    subgroup_stats = stats_df[stats_df["split"] != "Total"].drop(columns=["x_label"])
    if not subgroup_stats.empty:
        print("\nBy opening / flow direction")
        print(subgroup_stats.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    return fig, axs, stats_df


fig, axs, stats_direct_network = plot_direct_network_comparison(
    "Direct Network",
    flowStatsMI_directNetwork,
    roomVentilationMI_directNetwork,
)


In [ ]:
optParams =  [6.647e-01, 6.558e-01, 5.395e-01, 6.928e-01, 2.923e-01, 1.106e-01, -1.354e-08, 1.775e-01]

fittedParams = {
    ('Window',   'Flow Entering'): np.array([optParams[0]/Cd, optParams[4]]),
    ('Window',   'Flow Exiting'):  np.array([optParams[1]/Cd, optParams[5]]),
    ('Skylight', 'Flow Entering'): np.array([optParams[2]/Cd, optParams[6]]),
    ('Skylight', 'Flow Exiting'):  np.array([optParams[3]/Cd, optParams[7]])
    }

def _params_for_opening(opening_type):
    group = "Skylight" if "skylight" in str(opening_type).lower() else "Window"
    return (
        [fittedParams[(group, "Flow Entering")][0]*Cd, fittedParams[(group, "Flow Exiting")][0]*Cd],
        [fittedParams[(group, "Flow Entering")][1], fittedParams[(group, "Flow Exiting")][1]],
    )

flowStatsMI[["C_d", "qRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

direct_network1_param_lookup = _direct_network_param_lookup_from_fitted_params(fittedParams)

flowStatsMI[["C_d", "qRMS"]]

In [ ]:
flowStatsMI.drop(columns=["pRMS"], inplace=True)
flowStatsMI_directNetwork1, roomVentilationMI_directNetwork1 = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [Cd, Cd], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
fig, axs, stats_direct_network1 = plot_direct_network_comparison(
    "$\sigma_{q_n}$ IN Bulk Fit",
    flowStatsMI_directNetwork1,
    roomVentilationMI_directNetwork1,
)

In [ ]:
optParams = [6.121e-01, 6.467e-01, 5.840e-01, 6.464e-01, 1.271e-01, 1.010e-01, 9.888e-09, 1.309e-01]

fittedParams = {
    ('Window',   'Flow Entering'): np.array([optParams[0]/Cd, optParams[4]]),
    ('Window',   'Flow Exiting'):  np.array([optParams[1]/Cd, optParams[5]]),
    ('Skylight', 'Flow Entering'): np.array([optParams[2]/Cd, optParams[6]]),
    ('Skylight', 'Flow Exiting'):  np.array([optParams[3]/Cd, optParams[7]])
    }

def _params_for_opening(opening_type):
    group = "Skylight" if "skylight" in str(opening_type).lower() else "Window"
    return (
        [fittedParams[(group, "Flow Entering")][0]*Cd, fittedParams[(group, "Flow Exiting")][0]*Cd],
        [fittedParams[(group, "Flow Entering")][1], fittedParams[(group, "Flow Exiting")][1]],
    )

flowStatsMI[["C_d", "pRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

direct_network2_param_lookup = _direct_network_param_lookup_from_fitted_params(fittedParams)

flowStatsMI[["C_d", "pRMS"]]

In [ ]:
flowStatsMI.drop(columns=["qRMS"], inplace=True)
flowStatsMI_directNetwork2, roomVentilationMI_directNetwork2 = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [Cd, Cd], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
fig, axs, stats_direct_network2 = plot_direct_network_comparison(
    "$\sigma_{\Delta C_p}$ IN Bulk Fit",
    flowStatsMI_directNetwork2,
    roomVentilationMI_directNetwork2,
)


In [ ]:
def _skylight_mask(frame):
    if "openingType" in frame.columns:
        return frame["openingType"].astype(str).str.contains("skylight", case=False, na=False)
    return frame["skylight"].fillna(0).astype(int) == 1

flow_direct_plot = flowStatsMI_directNetwork.copy()
flow_direct_plot["p-noInt-direct_optp0-q_model-Norm"] = (
    flow_direct_plot["p-noInt-direct_optp0-q_model"] / flow_direct_plot["WS"]
)
flow_direct_plot["flux-Norm"] = flow_direct_plot["flux"] / flow_direct_plot["WS"]

room_direct_plot = roomVentilationMI_directNetwork.copy()
room_direct_plot["p-noInt-direct_optp0-q_model-Norm"] = (
    room_direct_plot["p-noInt-direct_optp0-q_model"] / room_direct_plot["WS"]
)

flow_direct2_plot = flowStatsMI_directNetwork2.copy()
flow_direct2_plot["p-noInt-direct_optp0-q_model-Norm"] = (
    flow_direct2_plot["p-noInt-direct_optp0-q_model"] / flow_direct2_plot["WS"]
)
flow_direct2_plot["flux-Norm"] = flow_direct2_plot["flux"] / flow_direct2_plot["WS"]

room_direct2_plot = roomVentilationMI_directNetwork2.copy()
room_direct2_plot["p-noInt-direct_optp0-q_model-Norm"] = (
    room_direct2_plot["p-noInt-direct_optp0-q_model"] / room_direct2_plot["WS"]
)

slall_style_col = "Skylight State"
slall_style_order = ["Interiors Without Skylights", "Interiors With Skylights"]

def _add_slall_style(frame):
    frame = frame.copy()
    if "slAll" in frame.columns:
        frame[slall_style_col] = np.where(frame["slAll"].astype(bool), "Interiors With Skylights", "Interiors Without Skylights")
    else:
        frame[slall_style_col] = "Unknown"
    return frame

window_data = _add_slall_style(data[~_skylight_mask(data)])
skylight_data = _add_slall_style(data[_skylight_mask(data)])
window_direct_plot = _add_slall_style(flow_direct_plot[~_skylight_mask(flow_direct_plot)])
skylight_direct_plot = _add_slall_style(flow_direct_plot[_skylight_mask(flow_direct_plot)])
window_direct2_plot = _add_slall_style(flow_direct2_plot[~_skylight_mask(flow_direct2_plot)])
skylight_direct2_plot = _add_slall_style(flow_direct2_plot[_skylight_mask(flow_direct2_plot)])

non_slall_room_base = roomVentilationMI[roomVentilationMI["slAll"] == False]
slall_room_base = roomVentilationMI[roomVentilationMI["slAll"] == True]
non_slall_room_adj = roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == False]
slall_room_adj = roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == True]
non_slall_room_direct = room_direct_plot[room_direct_plot["slAll"] == False]
slall_room_direct = room_direct_plot[room_direct_plot["slAll"] == True]
non_slall_room_direct2 = room_direct2_plot[room_direct2_plot["slAll"] == False]
slall_room_direct2 = room_direct2_plot[room_direct2_plot["slAll"] == True]

def _opening_panel(title, frame, x_col):
    return {
        "title": title,
        "frame": frame,
        "x": x_col,
        "y": y_var,
        "hue": "roomType",
        "palette": "colorblind",
        "hue_order": room_type_order,
        "style": slall_style_col,
        "style_order": slall_style_order,
    }

def _room_panel(title, frame, x_col):
    return {
        "title": title,
        "frame": frame,
        "x": x_col,
        "y": y_var,
        "hue": "roomType",
        "palette": "colorblind",
        "hue_order": room_type_order,
        "style": "roomType",
        "style_order": room_type_order,
    }

panel_grid = [
    [
        _opening_panel("Window Baseline vs LES", window_data, x_var),
        _opening_panel("Window Adjusted vs LES", window_direct2_plot, xAdjusted),
        _opening_panel("Window Legacy Direct Network vs LES", window_direct_plot, "p-noInt-direct_optp0-q_model-Norm"),
        _opening_panel("Window $\sigma_{\Delta C_p}$ IN Bulk Fit vs LES", window_direct2_plot, "p-noInt-direct_optp0-q_model-Norm"),
    ],
    [
        _opening_panel("Skylight Baseline vs LES", skylight_data, x_var),
        _opening_panel("Skylight Adjusted vs LES", skylight_direct2_plot, xAdjusted),
        _opening_panel("Skylight Legacy Direct Network vs LES", skylight_direct_plot, "p-noInt-direct_optp0-q_model-Norm"),
        _opening_panel("Skylight $\sigma_{\Delta C_p}$ IN Bulk Fit vs LES", skylight_direct2_plot, "p-noInt-direct_optp0-q_model-Norm"),
    ],
    [
        _room_panel("Non-slAll Room Baseline vs LES", non_slall_room_base, x_var),
        _room_panel("Non-slAll Room Adjusted vs LES", non_slall_room_adj, "xAdjusted_room"),
        _room_panel("Non-slAll Room Legacy Direct Network vs LES", non_slall_room_direct, "p-noInt-direct_optp0-q_model-Norm"),
        _room_panel("Non-slAll Room $\sigma_{\Delta C_p}$ IN Bulk Fit vs LES", non_slall_room_direct2, "p-noInt-direct_optp0-q_model-Norm"),
    ],
    [
        _room_panel("slAll Room Baseline vs LES", slall_room_base, x_var),
        _room_panel("slAll Room Adjusted vs LES", slall_room_adj, "xAdjusted_room"),
        _room_panel("slAll Room Legacy Direct Network vs LES", slall_room_direct, "p-noInt-direct_optp0-q_model-Norm"),
        _room_panel("slAll Room $\sigma_{\Delta C_p}$ IN Bulk Fit vs LES", slall_room_direct2, "p-noInt-direct_optp0-q_model-Norm"),
    ],
]

fig, axs = plt.subplots(4, 4, figsize=(18, 15), dpi=160)

for panel_idx, (ax, spec) in enumerate(zip(axs.flatten(), [spec for row in panel_grid for spec in row])):
    row_idx, _ = divmod(panel_idx, len(panel_grid[0]))
    plot_frame, x_col = _panel_plot_data(spec["frame"], spec["x"])
    sns.scatterplot(
        data=plot_frame,
        x=x_col,
        y=spec["y"],
        hue=spec["hue"],
        style=spec["style"],
        palette=spec["palette"],
        hue_order=spec["hue_order"],
        style_order=spec["style_order"],
        ax=ax,
        alpha=0.7,
        s=20,
        legend=ax is axs[0, 0],
    )

    metrics = _comparison_metrics(spec["frame"], spec["x"], spec["y"])
    valid = plot_frame[[x_col, spec["y"]]].dropna()
    if not valid.empty:
        lim_min = valid.min().min()
        lim_max = valid.max().max()
        ax.plot([lim_min, lim_max], [lim_min, lim_max], "r--", linewidth=1.0, alpha=0.7)
        ax.set_xlim(lim_min, lim_max)
        ax.set_ylim(lim_min, lim_max)

    if row_idx < 2:
        _annotate_directional_r_squared(ax, spec["frame"], spec["x"], spec["y"], fontsize=10)

    ax.set_title(spec["title"], fontsize=12)
    ax.set_xlabel(metrics["x_label"], fontsize=10)
    ax.set_ylabel(spec["y"], fontsize=10)
    ax.grid(True, alpha=0.3)

handles, labels = axs[0, 0].get_legend_handles_labels()
for ax in axs.flatten():
    if ax.get_legend() is not None:
        ax.get_legend().remove()

if handles:
    fig.legend(
        handles,
        labels,
        loc="center left",
        bbox_to_anchor=(0.92, 0.5),
        frameon=False,
    )

fig.suptitle("Legacy Direct Network vs $\sigma_{\Delta C_p}$ IN Bulk Fit Comparison", fontsize=18)
fig.subplots_adjust(right=0.88, hspace=0.45, wspace=0.3)


In [ ]:
optParams = [6.882e-01, 7.107e-01, 6.848e-01, 7.639e-01]

fittedParams = {
    ('Window',   'Flow Entering'): np.array([optParams[0]/Cd, 0]),
    ('Window',   'Flow Exiting'):  np.array([optParams[1]/Cd, 0]),
    ('Skylight', 'Flow Entering'): np.array([optParams[2]/Cd, 0]),
    ('Skylight', 'Flow Exiting'):  np.array([optParams[3]/Cd, 0])
    }

flowStatsMI[["C_d", "pRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

direct_network3_param_lookup = _direct_network_param_lookup_from_fitted_params(fittedParams, measured_sigma=True)

flowStatsMI["pRMS"] = flowStatsMI.apply(lambda x: [x["p_rms-noInt"], x["p_rms-noInt"]], axis=1)

flowStatsMI[["C_d", "pRMS"]]

In [ ]:
flowStatsMI_directNetwork3, roomVentilationMI_directNetwork3 = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [Cd, Cd], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
fig, axs, stats_direct_network3 = plot_direct_network_comparison(
    "$\sigma_{\Delta C_p}$ IN Opening-Specific Measured",
    flowStatsMI_directNetwork3,
    roomVentilationMI_directNetwork3,
)


In [ ]:
# C_d fit in-network with sigma fixed to the bulk measured means (config2_post-multiRun_flowModel.ipynb, In-network Bulk Measured)
optParams = [6.882e-01, 6.894e-01, 6.703e-01, 7.380e-01]

p_rms_bulk_means = flowStatsMI.groupby(["skylight", "Sdelp"])["p_rms-noInt-Norm"].mean()

fittedParams = {
    ('Window',   'Flow Entering'): np.array([optParams[0]/Cd, p_rms_bulk_means[(0, 1)]]),
    ('Window',   'Flow Exiting'):  np.array([optParams[1]/Cd, p_rms_bulk_means[(0, -1)]]),
    ('Skylight', 'Flow Entering'): np.array([optParams[2]/Cd, p_rms_bulk_means[(1, 1)]]),
    ('Skylight', 'Flow Exiting'):  np.array([optParams[3]/Cd, p_rms_bulk_means[(1, -1)]])
    }

flowStatsMI[["C_d", "pRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

direct_network5_param_lookup = _direct_network_param_lookup_from_fitted_params(fittedParams)

# bulk sigma is WS-normalized, so scale back to Pa for each run
flowStatsMI["pRMS"] = flowStatsMI.apply(lambda x: [s * x["WS"]**2 for s in x["pRMS"]], axis=1)
flowStatsMI[["C_d", "pRMS"]]

In [ ]:
flowStatsMI_directNetwork5, roomVentilationMI_directNetwork5 = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [Cd, Cd], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
fig, axs, stats_direct_network5 = plot_direct_network_comparison(
    "$\sigma_{\Delta C_p}$ IN Bulk Measured",
    flowStatsMI_directNetwork5,
    roomVentilationMI_directNetwork5,
)

In [ ]:
optParams = [7.022e-01, 7.195e-01, 6.934e-01, 7.408e-01]

fittedParams = {
    ('Window',   'Flow Entering'): np.array([optParams[0]/Cd, 0]),
    ('Window',   'Flow Exiting'):  np.array([optParams[1]/Cd, 0]),
    ('Skylight', 'Flow Entering'): np.array([optParams[2]/Cd, 0]),
    ('Skylight', 'Flow Exiting'):  np.array([optParams[3]/Cd, 0])
    }

flowStatsMI[["C_d", "qRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

direct_network4_param_lookup = _direct_network_param_lookup_from_fitted_params(fittedParams, measured_sigma=True)

flowStatsMI["qRMS"] = flowStatsMI.apply(lambda x: [x["rms-mass_flux"], x["rms-mass_flux"]], axis=1)
flowStatsMI[["C_d", "qRMS"]]

In [ ]:
flowStatsMI.drop(columns=["pRMS"], inplace=True)
flowStatsMI_directNetwork4, roomVentilationMI_directNetwork4 = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [Cd, Cd], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
fig, axs, stats_direct_network4 = plot_direct_network_comparison(
    "$\sigma_{q_n}$ IN Opening-Specific Measured",
    flowStatsMI_directNetwork4,
    roomVentilationMI_directNetwork4,
)


In [ ]:
# C_d fit in-network with sigma fixed to the bulk measured means (config2_post-multiRun_flowModel.ipynb, In-network Bulk Measured)
optParams = [6.921e-01, 7.248e-01, 6.796e-01, 7.355e-01]

q_rms_bulk_means = flowStatsMI.groupby(["skylight", "Sdelp"])["rms-mass_flux-Norm"].mean()

fittedParams = {
    ('Window',   'Flow Entering'): np.array([optParams[0]/Cd, q_rms_bulk_means[(0, 1)]]),
    ('Window',   'Flow Exiting'):  np.array([optParams[1]/Cd, q_rms_bulk_means[(0, -1)]]),
    ('Skylight', 'Flow Entering'): np.array([optParams[2]/Cd, q_rms_bulk_means[(1, 1)]]),
    ('Skylight', 'Flow Exiting'):  np.array([optParams[3]/Cd, q_rms_bulk_means[(1, -1)]])
    }

flowStatsMI[["C_d", "qRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

direct_network6_param_lookup = _direct_network_param_lookup_from_fitted_params(fittedParams)

# bulk sigma is WS-normalized, so scale back for each run
flowStatsMI["qRMS"] = flowStatsMI.apply(lambda x: [s * x["WS"] for s in x["qRMS"]], axis=1)
flowStatsMI[["C_d", "qRMS"]]

In [ ]:
flowStatsMI.drop(columns=["pRMS"], inplace=True, errors="ignore")
flowStatsMI_directNetwork6, roomVentilationMI_directNetwork6 = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [Cd, Cd], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
fig, axs, stats_direct_network6 = plot_direct_network_comparison(
    "$\sigma_{q_n}$ IN Bulk Measured",
    flowStatsMI_directNetwork6,
    roomVentilationMI_directNetwork6,
)

In [ ]:
def _prepare_pointwise_in_frame(flow_frame):
    frame = flow_frame.copy()
    frame["p-noInt-direct_optp0-q_model-Norm"] = (
        frame["p-noInt-direct_optp0-q_model"] / frame["WS"]
    )
    frame["flux-Norm"] = frame["flux"] / frame["WS"]
    for measured_col in ["p_rms-noInt-Norm", "rms-mass_flux-Norm"]:
        if measured_col not in frame.columns:
            frame[measured_col] = data[measured_col].reindex(frame.index)
    frame["Cp"] = frame["p_rms-noInt-Norm"] / (rho / 2)
    return frame


pointwise_in_q = _prepare_pointwise_in_frame(flowStatsMI_directNetwork4)
pointwise_in_p = _prepare_pointwise_in_frame(flowStatsMI_directNetwork3)
pointwise_in_x = "p-noInt-direct_optp0-q_model-Norm"
pointwise_in_y = "flux-Norm"

fig_pointwise_in, axs_pointwise_in = plt.subplots(
    2, 2, figsize=(12, 8), dpi=160, sharex=True, sharey=True, layout="constrained"
)

pointwise_in_specs = [
    {
        "frame": pointwise_in_q,
        "hue": "rms-mass_flux-Norm",
        "norm": plt.Normalize(vmin=0, vmax=0.25),
        "palette": "plasma",
        "title": r"$\sigma_{q_n}$",
        "colorbar_label": r"$\sigma_{q_n}$",
        "colorbar_rect": [0.16, -0.1, 0.30, 0.018],
    },
    {
        "frame": pointwise_in_p,
        "hue": "Cp",
        "norm": plt.Normalize(vmin=0, vmax=0.25),
        "palette": "viridis",
        "title": r"$\sigma_{\Delta C_p}$",
        "colorbar_label": r"$\sigma_{\Delta C_p}$",
        "colorbar_rect": [0.61, -0.1, 0.30, 0.018],
    },
]

for ax in axs_pointwise_in.flatten():
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=16)
    ax.set_xlim(-0.6, 0.6)
    ax.set_ylim(-0.6, 0.6)

for col_idx, spec in enumerate(pointwise_in_specs):
    frame = spec["frame"].copy()
    if "skylight" not in frame.columns:
        frame["skylight"] = frame["openingType"].astype(str).str.contains(
            "skylight", case=False, na=False
        )
    frame = frame.sort_values(by=[spec["hue"], "slAll"])

    for row_idx, skylight_value in enumerate([False, True]):
        panel_data = frame[frame["skylight"].astype(bool) == skylight_value].copy()
        min_value = min(panel_data[pointwise_in_x].min(), panel_data[pointwise_in_y].min())
        max_value = max(panel_data[pointwise_in_x].max(), panel_data[pointwise_in_y].max())
        reference_values = np.array([min_value, max_value])
        axs_pointwise_in[row_idx, col_idx].plot(
            reference_values, reference_values, "k-", label="1:1 Line"
        )
        scatter_ax = sns.scatterplot(
            data=panel_data,
            x=pointwise_in_x,
            y=pointwise_in_y,
            hue=spec["hue"],
            style="slAll",
            palette=spec["palette"],
            hue_norm=spec["norm"],
            alpha=0.6,
            s=20,
            legend=False,
            ax=axs_pointwise_in[row_idx, col_idx],
        )
        if scatter_ax.collections:
            scatter_ax.collections[0].set_cmap(spec["palette"])
            scatter_ax.collections[0].set_norm(spec["norm"])
        _annotate_directional_r_squared(
            axs_pointwise_in[row_idx, col_idx],
            panel_data,
            pointwise_in_x,
            pointwise_in_y,
            fontsize=12,
        )

    axs_pointwise_in[0, col_idx].set_title(spec["title"], fontsize=18, pad=10)
    mappable = plt.cm.ScalarMappable(
        norm=spec["norm"], cmap=plt.get_cmap(spec["palette"])
    )
    mappable.set_array([])
    colorbar_ax = fig_pointwise_in.add_axes(spec["colorbar_rect"])
    colorbar = fig_pointwise_in.colorbar(mappable, cax=colorbar_ax, orientation="horizontal")
    colorbar.set_label(spec["colorbar_label"], fontsize=20)
    colorbar.ax.tick_params(labelsize=20)

for ax in axs_pointwise_in.flatten():
    ax.set_ylabel("")
    ax.set_xlabel("")

for row_idx, row_label in enumerate(["Windows", "Skylights"]):
    axs_pointwise_in[row_idx, 0].set_ylabel(row_label, fontsize=18, labelpad=12)

fig_pointwise_in.supxlabel("$q_{PW,n}$", fontsize=24, y=-0.05, x=0.53)
fig_pointwise_in.supylabel("LES $\overline{q_n}$", fontsize=24, x=-0.01, y=0.51)
fig_pointwise_in.subplots_adjust(
    left=0.12, right=0.94, top=0.90, bottom=0.2, wspace=0.2, hspace=0.22
)

metrics_iq_pointwise_in = _comparison_metrics(
    pointwise_in_q, pointwise_in_x, pointwise_in_y
)
metrics_ip_pointwise_in = _comparison_metrics(
    pointwise_in_p, pointwise_in_x, pointwise_in_y
)


In [ ]:
# In-network fluctuation intensity I and fluctuation correction s(I) for the opening-specific IN models.
# The network solve only stores the corrected flow, so back out I per opening (q_IN is monotonic in I):
#   sigma_dCp: q_IN = C_d A sqrt(p_rms / (rho I)) s_p(I),  with I = p_rms / (2|dP|)
#   sigma_q:   q_IN = (q_rms / I) s_q(I),                   with I = q_rms / |q_0| (q_0 = uncorrected orifice flow)
from scipy.optimize import brentq
from pyafn.ventilation import ventilationBlendedScaling_p, ventilationBlendedScaling_q


def in_network_intensity(in_flow_frame, model):
    q = in_flow_frame["p-noInt-direct_optp0-q_model"].to_numpy(float)
    if model == "p":
        C_d = np.array([cd[0] if q_i > 0 else cd[1] for cd, q_i in zip(in_flow_frame["C_d"], q)])
        p_rms = np.array([p[0] if q_i > 0 else p[1] for p, q_i in zip(in_flow_frame["pRMS"], q)])
        rms = p_rms
        target = np.abs(q) / (C_d * feUtils.A * np.sqrt(p_rms / rho))
        scaling = ventilationBlendedScaling_p
        flow_per_intensity = lambda I: scaling(np.atleast_1d(I))[0] / np.sqrt(I)
    else:
        q_rms = np.array([r[0] if q_i > 0 else r[1] for r, q_i in zip(in_flow_frame["qRMS"], q)])
        rms = q_rms
        target = np.abs(q) / q_rms
        scaling = ventilationBlendedScaling_q
        flow_per_intensity = lambda I: scaling(np.atleast_1d(I))[0] / I

    I = np.array([
        brentq(lambda I_i: flow_per_intensity(I_i) - t, 1e-10, 1e8) if t > 0 else np.inf
        for t in target
    ])
    correction = np.zeros_like(I)
    finite = np.isfinite(I)
    correction[finite] = scaling(I[finite])
    return pd.DataFrame({"I_IN": I, "correction_IN": correction, "rms_IN": rms}, index=in_flow_frame.index)


pointwise_in_q[["I_IN", "correction_IN", "rms_IN"]] = in_network_intensity(flowStatsMI_directNetwork4, "q")
pointwise_in_p[["I_IN", "correction_IN", "rms_IN"]] = in_network_intensity(flowStatsMI_directNetwork3, "p")


In [ ]:
# Opening-specific IN figure (rows: Window/Skylight; cols: q and p) colored by a shared in-network quantity
# (norm=None colors by the categorical room type instead, with a legend in place of the colorbar)
# Markers show the in-network correction regime from pyafn.ventilationScalingSwitch: mean-dominated (o) for
# I < I_crit and fluctuation-dominated (x) for I >= I_crit, with I_crit = 1/sqrt(2) (sigma_qn) and 1/sqrt(3) (sigma_dCp).
regime_markers = {"Mean-dominated": "o", "Fluctuation-dominated": "x"}  # drawn unfilled

def plot_pointwise_in_colored(hue, norm, palette, colorbar_label, x=pointwise_in_x, y=pointwise_in_y,
                              x_label="$q_{PW,n}$", y_label="LES $\\overline{q_n}$", colorbar_ticks=None):
    fig, axs = plt.subplots(2, 2, figsize=(9.6, 8), dpi=160, sharex=True, sharey=True, layout="constrained")
    specs = [
        (pointwise_in_q, r"$\sigma_{q_n}$", 1 / np.sqrt(2)),
        (pointwise_in_p, r"$\sigma_{\Delta C_p}$", 1 / np.sqrt(3)),
    ]

    for ax in axs.flatten():
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=16)
        ax.set_xlim(-0.6, 0.6)
        ax.set_ylim(-0.6, 0.6)

    for col_idx, (frame, title, I_crit) in enumerate(specs):
        frame = frame.copy()
        frame["regime"] = np.where(frame["I_IN"] >= I_crit, "Fluctuation-dominated", "Mean-dominated")
        if "skylight" not in frame.columns:
            frame["skylight"] = frame["openingType"].astype(str).str.contains(
                "skylight", case=False, na=False
            )
        frame = frame.sort_values(by=[hue, "slAll"])

        for row_idx, skylight_value in enumerate([False, True]):
            panel_data = frame[frame["skylight"].astype(bool) == skylight_value].copy()
            min_value = min(panel_data[x].min(), panel_data[y].min())
            max_value = max(panel_data[x].max(), panel_data[y].max())
            reference_values = np.array([min_value, max_value])
            axs[row_idx, col_idx].plot(reference_values, reference_values, "r--", label="1:1 Line")
            for regime, marker in regime_markers.items():
                regime_data = panel_data[panel_data["regime"] == regime]
                if regime_data.empty:
                    continue
                if norm is None:
                    colors = [palette[value] for value in regime_data[hue]]
                else:
                    colors = plt.get_cmap(palette)(norm(regime_data[hue].to_numpy(float)))
                axs[row_idx, col_idx].scatter(
                    regime_data[x],
                    regime_data[y],
                    marker=marker,
                    s=15,
                    linewidths=1.0,
                    alpha=0.8,
                    **({"facecolors": "none", "edgecolors": colors} if marker == "o" else {"c": colors}),
                )
            _annotate_directional_r_squared(
                axs[row_idx, col_idx], panel_data, x, y, fontsize=14
            )

        axs[0, col_idx].set_title(title, fontsize=18, pad=10)

    if norm is None:
        room_handles = [
            Line2D([0], [0], color=palette[room_type], marker="o", linestyle="None",
                    markersize=4, label=room_type_display_labels[room_type])
            for room_type in room_type_order
        ]
    else:
        mappable = plt.cm.ScalarMappable(norm=norm, cmap=plt.get_cmap(palette))
        mappable.set_array([])
        colorbar_ax = fig.add_axes([1.01, 0.08, 0.02, 0.84])
        colorbar = fig.colorbar(mappable, cax=colorbar_ax, orientation="vertical")
        colorbar.set_label(colorbar_label, fontsize=20)
        if colorbar_ticks is not None:
            colorbar.set_ticks(colorbar_ticks)
            colorbar.set_ticklabels([f"{tick:g}" for tick in colorbar_ticks])
            colorbar.ax.minorticks_off()
        colorbar.ax.tick_params(labelsize=20)

    regime_handles = [
        Line2D([0], [0], color="0.3", marker=marker, markerfacecolor="none", linestyle="None",
               markersize=4, markeredgewidth=1.0, label=regime)
        for regime, marker in regime_markers.items()
    ]
    legend_handles = (room_handles if norm is None else []) + regime_handles
    # Top-left panel, just below its "Flow in" R^2 label
    axs[0, 0].legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(0.0, 0.89),
                     fontsize=14, frameon=True, facecolor="white", edgecolor="0.6", framealpha=0.9,
                     fancybox=False, handletextpad=0.3, borderpad=0.3, labelspacing=0.3)

    for ax in axs.flatten():
        ax.set_ylabel("")
        ax.set_xlabel("")

    for row_idx, row_label in enumerate(["Windows", "Skylights"]):
        axs[row_idx, 0].set_ylabel(row_label, fontsize=18, labelpad=-6)

    fig.supxlabel(x_label, fontsize=24, y=-0.05, x=0.53)
    fig.supylabel(y_label, fontsize=24, x=-0.036, y=0.51)  # -0.01 shifted left 18 pt to offset the row-label labelpad change
    return fig, axs


fig_pointwise_in_corr, axs_pointwise_in_corr = plot_pointwise_in_colored(
    hue="correction_IN",
    norm=plt.Normalize(vmin=0, vmax=1),
    palette="viridis",
    colorbar_label="$F_{fc}(I)$",
)


In [ ]:
# Same opening-specific IN figure colored by the in-network fluctuation intensity I
from matplotlib.colors import LogNorm

fig_pointwise_in_I, axs_pointwise_in_I = plot_pointwise_in_colored(
    hue="I_IN",
    norm=LogNorm(vmin=0.1, vmax=10),
    palette="plasma",
    colorbar_label="In-Network $I$",
)


In [ ]:
# Bulk vs opening-specific in-network fluctuation correction per opening (rows: Window/Skylight; cols: q and p).
# Bulk uses the subgroup-mean sigma, opening-specific the opening's measured sigma. Points near 1:1 mean s(I) is set
# mainly by the mean flow / mean pressure, with opening-to-opening sigma variations a secondary effect.
from matplotlib.colors import LogNorm

bulk_in_q = in_network_intensity(flowStatsMI_directNetwork6, "q")
bulk_in_p = in_network_intensity(flowStatsMI_directNetwork5, "p")

correction_specs = [
    (pointwise_in_q, bulk_in_q, r"$\sigma_{q_n}$"),
    (pointwise_in_p, bulk_in_p, r"$\sigma_{\Delta C_p}$"),
]
sigma_ratio_norm = LogNorm(vmin=0.25, vmax=4)

fig_correction_compare, axs_correction_compare = plt.subplots(
    2, 2, figsize=(12, 10), dpi=160, sharex=True, sharey=True, layout="constrained"
)

for col_idx, (pointwise_frame, bulk_frame, title) in enumerate(correction_specs):
    frame = pd.DataFrame({
        "bulk": bulk_frame["correction_IN"],
        "opening-specific": pointwise_frame["correction_IN"],
        "sigma_ratio": pointwise_frame["rms_IN"] / bulk_frame["rms_IN"],
        "Sdelp": pointwise_frame["Sdelp"],
        "skylight": pointwise_frame["openingType"].astype(str).str.contains("skylight", case=False, na=False),
    }).dropna()
    frame = frame.iloc[np.argsort(np.abs(np.log(frame["sigma_ratio"].to_numpy())))]

    for row_idx, skylight_value in enumerate([False, True]):
        ax = axs_correction_compare[row_idx, col_idx]
        panel_data = frame[frame["skylight"] == skylight_value]
        ax.plot([0, 1], [0, 1], "k-", label="1:1 Line")
        ax.scatter(
            panel_data["bulk"],
            panel_data["opening-specific"],
            c=panel_data["sigma_ratio"],
            cmap="coolwarm",
            norm=sigma_ratio_norm,
            s=12,
            alpha=0.7,
            linewidths=0,
        )
        _annotate_directional_r_squared(ax, panel_data, "bulk", "opening-specific", fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=16)
        ax.set_xlim(0, 1.02)
        ax.set_ylim(0, 1.02)
        ax.set_aspect("equal")

    axs_correction_compare[0, col_idx].set_title(title, fontsize=18, pad=10)
    print(title, "overall R^2 (opening-specific vs bulk):", round(_comparison_metrics(frame, "bulk", "opening-specific")["r_squared"], 3))

for row_idx, row_label in enumerate(["Windows", "Skylights"]):
    axs_correction_compare[row_idx, 0].set_ylabel(row_label, fontsize=18, labelpad=12)

mappable = plt.cm.ScalarMappable(norm=sigma_ratio_norm, cmap="coolwarm")
mappable.set_array([])
colorbar_ax = fig_correction_compare.add_axes([1.01, 0.08, 0.02, 0.84])
colorbar = fig_correction_compare.colorbar(mappable, cax=colorbar_ax, orientation="vertical")
colorbar.set_label(r"$\sigma_{\mathrm{opening\text{-}specific}} / \sigma_{\mathrm{bulk}}$", fontsize=20)
colorbar.set_ticks([0.25, 0.5, 1, 2, 4])
colorbar.set_ticklabels(["0.25", "0.5", "1", "2", "4"])
colorbar.ax.minorticks_off()
colorbar.ax.tick_params(labelsize=16)

fig_correction_compare.supxlabel("Bulk fluctuation correction $s(I)$", fontsize=22, y=-0.03)
fig_correction_compare.supylabel("Opening-specific fluctuation correction $s(I)$", fontsize=22, x=-0.06)


In [ ]:
# Same opening-specific IN figure colored by the ratio of opening-specific to bulk in-network fluctuation correction
from matplotlib.colors import LogNorm

pointwise_in_q["correction_ratio"] = pointwise_in_q["correction_IN"] / bulk_in_q["correction_IN"]
pointwise_in_p["correction_ratio"] = pointwise_in_p["correction_IN"] / bulk_in_p["correction_IN"]

fig_pointwise_in_ratio, axs_pointwise_in_ratio = plot_pointwise_in_colored(
    hue="correction_ratio",
    norm=LogNorm(vmin=0.5, vmax=2),
    palette="coolwarm",
    colorbar_label=r"$s_{\mathrm{opening\text{-}specific}}(I) / s_{\mathrm{bulk}}(I)$",
    colorbar_ticks=[0.5, 0.7, 1, 1.4, 2],
)


In [ ]:
# In-network bulk vs opening-specific q_PW per opening (rows: Window/Skylight; cols: q and p), colored by LES ventilation rate
pointwise_in_q["q_PW_bulk"] = _prepare_pointwise_in_frame(flowStatsMI_directNetwork6)[
    "p-noInt-direct_optp0-q_model-Norm"
].reindex(pointwise_in_q.index)
pointwise_in_p["q_PW_bulk"] = _prepare_pointwise_in_frame(flowStatsMI_directNetwork5)[
    "p-noInt-direct_optp0-q_model-Norm"
].reindex(pointwise_in_p.index)

fig_bulk_vs_pointwise_in, axs_bulk_vs_pointwise_in = plot_pointwise_in_colored(
    hue=pointwise_in_y,
    norm=plt.Normalize(vmin=-0.5, vmax=0.5),
    palette="coolwarm",
    colorbar_label=r"LES $\overline{q_n}$",
    x="q_PW_bulk",
    y=pointwise_in_x,
    x_label="In-Network Bulk $q_{PW,n}$",
    y_label="In-Network Opening-Specific $q_{PW,n}$",
)


In [ ]:
# In-network bulk vs opening-specific q_PW per opening (rows: Window/Skylight; cols: q and p), colored by room type
fig_bulk_vs_pointwise_in_room, axs_bulk_vs_pointwise_in_room = plot_pointwise_in_colored(
    hue="roomType",
    norm=None,
    palette=room_type_palette,
    colorbar_label=None,
    x="q_PW_bulk",
    y=pointwise_in_x,
    x_label="In-Network Bulk $q_{PW,n}$",
    y_label="In-Network Opening-Specific $q_{PW,n}$",
)


In [ ]:
# In-network opening-specific vs bulk q_PW per opening (rows: Window/Skylight; cols: q and p), colored by sigma_bulk - sigma_opening-specific.
# Differences are nondimensionalized like the model inputs: sigma_qn = q_rms / WS and sigma_dCp = p_rms / (rho/2 WS^2).
pointwise_in_q["sigma_diff"] = (bulk_in_q["rms_IN"] - pointwise_in_q["rms_IN"]) / pointwise_in_q["WS"]
pointwise_in_p["sigma_diff"] = (bulk_in_p["rms_IN"] - pointwise_in_p["rms_IN"]) / (rho / 2 * pointwise_in_p["WS"] ** 2)

fig_bulk_vs_pointwise_in_sigma, axs_bulk_vs_pointwise_in_sigma = plot_pointwise_in_colored(
    hue="sigma_diff",
    norm=plt.Normalize(vmin=-0.1, vmax=0.1),
    palette="coolwarm",
    colorbar_label=r"$\sigma_{\mathrm{bulk}} - \sigma_{\mathrm{opening\text{-}specific}}$",
    x=pointwise_in_x,
    y="q_PW_bulk",
    x_label="Opening-Specific $q_{PW,n}$",
    y_label="Bulk $q_{PW,n}$",
)

# Bulk regime transitions: the bulk q_PW,n at which the bulk I crosses I*. The bulk networks use one C_d and
# WS-normalized sigma per (opening group, direction), so from the flow relations in in_network_intensity:
#   sigma_qn:  q_PW,n* = sigma s_q(I*) / I*                        (sigma = q_rms / WS)
#   sigma_dCp: q_PW,n* = C_d A sqrt(sigma / (rho I*)) s_p(I*)      (sigma = p_rms / WS^2)
bulk_transition_specs = [
    (direct_network6_param_lookup, 1 / np.sqrt(2),
     lambda cd, sigma, I_star: sigma * ventilationBlendedScaling_q(np.atleast_1d(I_star))[0] / I_star),
    (direct_network5_param_lookup, 1 / np.sqrt(3),
     lambda cd, sigma, I_star: cd * feUtils.A * np.sqrt(sigma / (rho * I_star)) * ventilationBlendedScaling_p(np.atleast_1d(I_star))[0]),
]
for col_idx, (param_lookup, I_star, q_star_from_params) in enumerate(bulk_transition_specs):
    for row_idx, opening_group in enumerate(["Window", "Skylight"]):
        ax = axs_bulk_vs_pointwise_in_sigma[row_idx, col_idx]
        for direction, sign in [("Flow Entering", 1), ("Flow Exiting", -1)]:
            params = param_lookup[(opening_group, direction)]
            y_star = sign * q_star_from_params(params["cd"], params["sigma"], I_star)
            if not ax.get_ylim()[0] < y_star < ax.get_ylim()[1]:
                continue
            ax.axhline(y_star, color="0.3", linestyle="-.", linewidth=1, zorder=1)
            # Label at the right end, on the side of the line away from the origin, in the first panel only
            if (row_idx, col_idx) == (0, 0):
                ax.annotate(
                    r"$q_{PW,n}(I^*_{\mathrm{bulk}})$", (0.97, y_star), xycoords=ax.get_yaxis_transform(),
                    xytext=(0, 3 * sign), textcoords="offset points", ha="right", va="bottom" if sign > 0 else "top",
                    fontsize=14, color="0.3",
                )

In [ ]:
# Same opening-specific IN figure colored by the opening-specific minus bulk in-network q_PW
pointwise_in_q["q_PW_diff"] = pointwise_in_q[pointwise_in_x] - pointwise_in_q["q_PW_bulk"]
pointwise_in_p["q_PW_diff"] = pointwise_in_p[pointwise_in_x] - pointwise_in_p["q_PW_bulk"]

fig_pointwise_in_diff, axs_pointwise_in_diff = plot_pointwise_in_colored(
    hue="q_PW_diff",
    norm=plt.Normalize(vmin=-0.1, vmax=0.1),
    palette="coolwarm",
    colorbar_label=r"$q_{PW,n}^{\mathrm{opening\text{-}specific}} - q_{PW,n}^{\mathrm{bulk}}$",
)


In [ ]:

def _ensure_iqr_error_column(df):
    if "iqr_error" in df.columns:
        return df
    df = df.copy()
    for candidate in ["iqr", "error_iqr"]:
        if candidate in df.columns:
            df["iqr_error"] = df[candidate]
            return df
    df["iqr_error"] = np.nan
    return df


def _ensure_r_squared_column(df):
    if "r_squared" in df.columns:
        return df
    df = df.copy()
    df["r_squared"] = 1 - df["nrmse"] ** 2
    return df


def _apply_direct_network_params(df, param_lookup):
    if param_lookup is None:
        return df

    df = df.copy()
    for (opening_group, direction), params in param_lookup.items():
        mask = (df["opening_group"] == opening_group) & (df["direction"] == direction)
        df.loc[mask, "cd"] = params["cd"]
        df.loc[mask, "sigma"] = params["sigma"]
    return df


def _standardize_fit_metrics(metrics_df):
    df = _ensure_r_squared_column(_ensure_iqr_error_column(metrics_df))
    df["source"] = "plot_ventilation_model_fit"
    df["run_label"] = df["model_name"]
    return df[
        [
            "source",
            "run_label",
            "model_name",
            "scope",
            "opening_group",
            "direction",
            "split",
            "adjusted",
            "cd",
            "sigma",
            "rmse",
            "nrmse",
            "r_squared",
            "bias",
            "iqr_error",
            "std",
            "n",
            "x_label",
            "y_label",
        ]
    ]


def _standardize_direct_network_metrics(run_label, stats_df, param_lookup=None):
    df = _ensure_r_squared_column(_ensure_iqr_error_column(stats_df))
    split_parts = df["split"].str.split(",", n=1, expand=True)
    df["source"] = "direct_network"
    df["run_label"] = run_label
    df["model_name"] = df["panel"]
    df["scope"] = np.where(df["split"].eq("Total"), "panel_total", "subgroup")
    df["opening_group"] = np.where(df["split"].eq("Total"), pd.NA, split_parts[0])
    df["direction"] = np.where(df["split"].eq("Total"), "All", split_parts[1].str.strip())
    df["adjusted"] = df["panel"].str.contains("Adjusted", na=False)
    df["cd"] = np.nan
    df["sigma"] = np.nan
    df = _apply_direct_network_params(df, param_lookup)
    df["y_label"] = y_var
    return df[
        [
            "source",
            "run_label",
            "model_name",
            "scope",
            "opening_group",
            "direction",
            "split",
            "adjusted",
            "cd",
            "sigma",
            "rmse",
            "nrmse",
            "r_squared",
            "bias",
            "iqr_error",
            "std",
            "n",
            "x_label",
            "y_label",
        ]
    ]


def _room_slall_metrics(run_label, model_name, frame, x_spec, source, adjusted=False):
    rows = []
    room_type_specs = [("All", pd.Series(True, index=frame.index))]
    if "roomType" in frame.columns:
        for room_type in room_type_order:
            room_type_specs.append((room_type, frame["roomType"] == room_type))

    for sl_all, opening_group in [(False, "Window"), (True, "Skylight")]:
        sl_mask = frame["slAll"] == sl_all
        for room_type, room_type_mask in room_type_specs:
            subset = frame[sl_mask & room_type_mask]
            metrics = _comparison_metrics(subset, x_spec, y_var)
            if metrics["n"] == 0:
                continue
            split_label = f"{opening_group} Rooms" if room_type == "All" else f"{opening_group} {room_type} Rooms"
            rows.append(
                {
                    "source": source,
                    "run_label": run_label,
                    "model_name": model_name,
                    "scope": "room_slall",
                    "opening_group": opening_group,
                    "room_type": room_type,
                    "direction": "All",
                    "split": split_label,
                    "adjusted": adjusted,
                    "cd": np.nan,
                    "sigma": np.nan,
                    "rmse": metrics["rmse"],
                    "nrmse": metrics["nrmse"],
                    "r_squared": metrics["r_squared"],
                    "bias": metrics["bias"],
                    "iqr_error": metrics["iqr_error"],
                    "std": metrics["std"],
                    "n": metrics["n"],
                    "x_label": metrics["x_label"],
                    "y_label": y_var,
                }
            )
    return pd.DataFrame(rows)


def _model_spec_for_room_aggregation(model_name):
    if "Opening-Specific" in model_name and "\sigma_{q_n}" in model_name:
        return ventilationReDecomp_qMeasured, x_var2_q
    if "Opening-Specific" in model_name and "\sigma_{\Delta C_p}" in model_name:
        return ventilationReDecomp_pMeasured, x_var2_p
    if "\sigma_{\Delta C_p}" in model_name:
        return pyafn.ventilationReDecomp_p, None
    return pyafn.ventilationReDecomp_q, None


def _opening_predictions_from_fit_metrics(metrics_df):
    subgroup_metrics = metrics_df[metrics_df["scope"] == "subgroup"].copy()
    if subgroup_metrics.empty:
        return pd.Series(dtype=float)

    model_name = subgroup_metrics["model_name"].iloc[0]
    model_func, x_var2_col = _model_spec_for_room_aggregation(model_name)
    prediction = pd.Series(np.nan, index=data.index, dtype=float, name=f"{model_name}_opening_prediction")

    for _, metric_row in subgroup_metrics.iterrows():
        sl_val = metric_row["opening_group"] == "Skylight"
        sdelp = 1 if metric_row["direction"] == "Flow Entering" else -1
        mask = (data["skylight"].astype(bool) == sl_val) & (data["Sdelp"] == sdelp)
        if not mask.any():
            continue

        popt = [metric_row["cd"] / Cd, metric_row["sigma"]]
        x_abs = np.abs(data.loc[mask, x_var].values)
        if x_var2_col is None:
            y_pred = model_func(x_abs, *popt)
        else:
            x2_abs = np.abs(data.loc[mask, x_var2_col].values)
            y_pred = model_func((x_abs, x2_abs), *popt)
        prediction.loc[mask] = y_pred * sdelp

    return prediction


def _room_metrics_from_fit_metrics(metrics_df):
    model_name = metrics_df["model_name"].iloc[0]
    opening_prediction = _opening_predictions_from_fit_metrics(metrics_df)
    room_prediction_col = f"{model_name}_room_prediction"
    room_prediction_frame = aggregate_window_series_to_room(
        roomVentilationMI,
        opening_prediction,
        out_col=room_prediction_col,
        mode="abs_half_sum",
    )
    return _room_slall_metrics(
        model_name,
        model_name,
        room_prediction_frame,
        room_prediction_col,
        source="plot_ventilation_model_fit",
        adjusted=bool(metrics_df["adjusted"].any()),
    )


def _direct_network_room_slall_metrics(run_label, room_frame):
    room_plot = room_frame.copy()
    direct_col = "p-noInt-direct_optp0-q_model-Norm"
    room_plot[direct_col] = room_plot["p-noInt-direct_optp0-q_model"] / room_plot["WS"]

    return pd.concat(
        [
            _room_slall_metrics(run_label, "Room Baseline vs LES", roomVentilationMI, x_var, "direct_network"),
            _room_slall_metrics(run_label, "Room Adjusted vs LES", roomVentilationMI_adj, "xAdjusted_room", "direct_network", adjusted=True),
            _room_slall_metrics(run_label, f"Room {run_label} vs LES", room_plot, direct_col, "direct_network"),
        ],
        ignore_index=True,
    )


def _direct_network_i_type_metrics(run_label, i_model_name, stats_df, room_frame, param_lookup=None):
    opening_stats = _standardize_direct_network_metrics(run_label, stats_df, param_lookup)
    opening_stats = opening_stats[
        opening_stats["model_name"].isin([
            f"Window {run_label} vs LES",
            f"Skylight {run_label} vs LES",
        ])
        & (opening_stats["scope"] == "subgroup")
    ].copy()

    room_stats = _direct_network_room_slall_metrics(run_label, room_frame)
    room_stats = room_stats[room_stats["model_name"] == f"Room {run_label} vs LES"].copy()

    direct_stats = pd.concat([opening_stats, room_stats], ignore_index=True)
    direct_stats["source"] = "direct_network_i_type"
    direct_stats["run_label"] = i_model_name
    direct_stats["model_name"] = i_model_name
    direct_stats = _apply_direct_network_params(direct_stats, param_lookup)
    return direct_stats


fit_metric_frames = [
    metrics_ps_baseline,
    metrics_ps_fit,
    metrics_iq_fit,
    metrics_iq_bulk,
    metrics_ip_fit,
    metrics_ip_bulk,
    metrics_iq_pointwise,
    metrics_ip_pointwise,
]

direct_network_param_lookups = {
    "$\sigma_{q_n}$ IN Bulk Fit": globals().get("direct_network1_param_lookup"),
    "$\sigma_{\Delta C_p}$ IN Bulk Fit": globals().get("direct_network2_param_lookup"),
    "$\sigma_{\Delta C_p}$ IN Opening-Specific Measured": globals().get("direct_network3_param_lookup"),
    "$\sigma_{\Delta C_p}$ IN Bulk Measured": globals().get("direct_network5_param_lookup"),
    "$\sigma_{q_n}$ IN Opening-Specific Measured": globals().get("direct_network4_param_lookup"),
    "$\sigma_{q_n}$ IN Bulk Measured": globals().get("direct_network6_param_lookup"),
}

direct_network_metric_frames = [
    _standardize_direct_network_metrics("$\sigma_{q_n}$ IN Bulk Fit", stats_direct_network1, direct_network_param_lookups["$\sigma_{q_n}$ IN Bulk Fit"]),
    _standardize_direct_network_metrics("$\sigma_{\Delta C_p}$ IN Bulk Fit", stats_direct_network2, direct_network_param_lookups["$\sigma_{\Delta C_p}$ IN Bulk Fit"]),
    _standardize_direct_network_metrics("$\sigma_{\Delta C_p}$ IN Opening-Specific Measured", stats_direct_network3, direct_network_param_lookups["$\sigma_{\Delta C_p}$ IN Opening-Specific Measured"]),
    _standardize_direct_network_metrics("$\sigma_{\Delta C_p}$ IN Bulk Measured", stats_direct_network5, direct_network_param_lookups["$\sigma_{\Delta C_p}$ IN Bulk Measured"]),
    _standardize_direct_network_metrics("$\sigma_{q_n}$ IN Opening-Specific Measured", stats_direct_network4, direct_network_param_lookups["$\sigma_{q_n}$ IN Opening-Specific Measured"]),
    _standardize_direct_network_metrics("$\sigma_{q_n}$ IN Bulk Measured", stats_direct_network6, direct_network_param_lookups["$\sigma_{q_n}$ IN Bulk Measured"]),
]

fit_room_metric_frames = [_room_metrics_from_fit_metrics(df) for df in fit_metric_frames]

direct_network_room_metric_frames = [
    _direct_network_room_slall_metrics("$\sigma_{q_n}$ IN Bulk Fit", roomVentilationMI_directNetwork1),
    _direct_network_room_slall_metrics("$\sigma_{\Delta C_p}$ IN Bulk Fit", roomVentilationMI_directNetwork2),
    _direct_network_room_slall_metrics("$\sigma_{\Delta C_p}$ IN Opening-Specific Measured", roomVentilationMI_directNetwork3),
    _direct_network_room_slall_metrics("$\sigma_{\Delta C_p}$ IN Bulk Measured", roomVentilationMI_directNetwork5),
    _direct_network_room_slall_metrics("$\sigma_{q_n}$ IN Opening-Specific Measured", roomVentilationMI_directNetwork4),
    _direct_network_room_slall_metrics("$\sigma_{q_n}$ IN Bulk Measured", roomVentilationMI_directNetwork6),
]

direct_network_i_type_metric_frames = [
    _direct_network_i_type_metrics(
        "$\sigma_{q_n}$ IN Bulk Fit",
        "$\sigma_{q_n}$ IN Bulk Fit",
        stats_direct_network1,
        roomVentilationMI_directNetwork1,
        direct_network_param_lookups["$\sigma_{q_n}$ IN Bulk Fit"],
    ),
    _direct_network_i_type_metrics(
        "$\sigma_{q_n}$ IN Opening-Specific Measured",
        "$\sigma_{q_n}$ IN Opening-Specific Measured",
        stats_direct_network4,
        roomVentilationMI_directNetwork4,
        direct_network_param_lookups["$\sigma_{q_n}$ IN Opening-Specific Measured"],
    ),
    _direct_network_i_type_metrics(
        "$\sigma_{q_n}$ IN Bulk Measured",
        "$\sigma_{q_n}$ IN Bulk Measured",
        stats_direct_network6,
        roomVentilationMI_directNetwork6,
        direct_network_param_lookups["$\sigma_{q_n}$ IN Bulk Measured"],
    ),
    _direct_network_i_type_metrics(
        "$\sigma_{\Delta C_p}$ IN Bulk Fit",
        "$\sigma_{\Delta C_p}$ IN Bulk Fit",
        stats_direct_network2,
        roomVentilationMI_directNetwork2,
        direct_network_param_lookups["$\sigma_{\Delta C_p}$ IN Bulk Fit"],
    ),
    _direct_network_i_type_metrics(
        "$\sigma_{\Delta C_p}$ IN Opening-Specific Measured",
        "$\sigma_{\Delta C_p}$ IN Opening-Specific Measured",
        stats_direct_network3,
        roomVentilationMI_directNetwork3,
        direct_network_param_lookups["$\sigma_{\Delta C_p}$ IN Opening-Specific Measured"],
    ),
    _direct_network_i_type_metrics(
        "$\sigma_{\Delta C_p}$ IN Bulk Measured",
        "$\sigma_{\Delta C_p}$ IN Bulk Measured",
        stats_direct_network5,
        roomVentilationMI_directNetwork5,
        direct_network_param_lookups["$\sigma_{\Delta C_p}$ IN Bulk Measured"],
    ),
]

error_metrics_tidy = pd.concat(
    [_standardize_fit_metrics(df) for df in fit_metric_frames]
    + direct_network_metric_frames
    + fit_room_metric_frames
    + direct_network_room_metric_frames
    + direct_network_i_type_metric_frames,
    ignore_index=True,
)

error_metrics_tidy = error_metrics_tidy.sort_values(
    ["source", "run_label", "scope", "opening_group", "direction"]
).reset_index(drop=True)

display(error_metrics_tidy)


In [ ]:
# In-network vs post-network sigma_dCp bulk opening predictions (1:1; panels: openings per room)
openings_per_room = {
    (False, "corner"): 2, (False, "cross"): 2, (False, "dual"): 3,
    (True, "single"): 2, (True, "corner"): 3, (True, "cross"): 4, (True, "dual"): 5,
}
opening_count_grid = [[2, 3], [4, 5]]
bulk_openings_x = "IN Bulk"
bulk_openings_y = "PN Bulk"

bulk_openings = data[["roomType", "slAll", "Sdelp"]].copy()
bulk_openings["n_openings"] = [
    openings_per_room.get((bool(sl_all), room_type), np.nan)
    for sl_all, room_type in zip(bulk_openings["slAll"], bulk_openings["roomType"])
]
bulk_openings[bulk_openings_y] = _opening_predictions_from_fit_metrics(metrics_ip_bulk)
bulk_openings[bulk_openings_x] = _prepare_pointwise_in_frame(flowStatsMI_directNetwork5)[
    "p-noInt-direct_optp0-q_model-Norm"
].reindex(bulk_openings.index)
bulk_openings = bulk_openings.dropna(subset=[bulk_openings_x, bulk_openings_y, "n_openings"])

fig_bulk_openings, axs_bulk_openings = plt.subplots(
    2, 2, figsize=(9.6, 8), dpi=160, sharex=True, sharey=True, layout="constrained"
)

for row_idx, opening_counts in enumerate(opening_count_grid):
    for col_idx, n_openings in enumerate(opening_counts):
        ax = axs_bulk_openings[row_idx, col_idx]
        panel_data = bulk_openings[bulk_openings["n_openings"] == n_openings]

        ax.plot([-0.6, 0.6], [-0.6, 0.6], "r--", linewidth=1.2, zorder=0)
        for room_type in room_type_order:
            room_data = panel_data[panel_data["roomType"] == room_type]
            if room_data.empty:
                continue
            ax.scatter(
                room_data[bulk_openings_x],
                room_data[bulk_openings_y],
                color=room_type_palette[room_type],
                s=4.5,
                linewidths=0,
            )

        for mask, label, position, ha, va, color in [
            (panel_data["Sdelp"] > 0, "Flow in", (0.03, 0.97), "left", "top", "tab:red"),
            (panel_data["Sdelp"] < 0, "Flow out", (0.97, 0.03), "right", "bottom", "tab:blue"),
        ]:
            metrics = _comparison_metrics(panel_data.loc[mask], bulk_openings_x, bulk_openings_y)
            ax.text(
                *position,
                rf"{label}: $R^2={metrics['r_squared']:.3f}$",
                transform=ax.transAxes,
                ha=ha,
                va=va,
                fontsize=14,
                color=color,
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.75, "pad": 1.5},
                zorder=5,
            )

        # Room-type legend for the rooms in this panel, just below its "Flow in" R^2 label
        panel_rooms = set(zip(panel_data["roomType"], panel_data["slAll"].astype(bool)))
        panel_room_handles = [
            Line2D([0], [0], color=room_type_palette[room_type], marker="o", linestyle="None", markersize=8,
                   label=room_type_display_labels[room_type] + (" (with skylights)" if sl_all else ""))
            for room_type in room_type_order
            for sl_all in (False, True)
            if (room_type, sl_all) in panel_rooms
        ]
        ax.legend(handles=panel_room_handles, loc="upper left", bbox_to_anchor=(0.0, 0.89),
                  fontsize=14, frameon=True, facecolor="white", edgecolor="0.6", framealpha=0.9,
                  fancybox=False, handletextpad=0.3, borderpad=0.3, labelspacing=0.3)

        ax.set_title(f"{n_openings} Openings", fontsize=18, pad=10)
        ax.set_xlim(-0.6, 0.6)
        ax.set_ylim(-0.6, 0.6)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=16)

fig_bulk_openings.supxlabel("In-Network $q_{PW,n}$", fontsize=24, y=-0.05, x=0.53)
fig_bulk_openings.supylabel("Post-Network $q_{PW,n}$", fontsize=24, x=-0.045, y=0.51)


In [ ]:
import os

ashrae_baseline_label = "ASHRAE Mean-Pressure Baseline"
ashrae_q_label = r"ASHRAE $\sigma_{q_n}$ PN Bulk Fit"
ashrae_p_label = r"ASHRAE $\sigma_{\Delta C_p}$ PN Bulk Fit"
ashrae_error_metrics_path = os.path.join(home_dir, "ashrae_exports", "ashrae_error_metrics.csv")
ashrae_fit_params_path = os.path.join(home_dir, "ashrae_exports", "ashrae_fit_params.csv")

original_table_model_order = [
    ashrae_baseline_label,
    ashrae_q_label,
    ashrae_p_label,
    "Mean-Pressure Baseline",
    "Mean-Pressure Fit PN",
    "$\sigma_{q_n}$ PN Bulk Fit",
    "$\sigma_{q_n}$ PN Bulk Measured",
    "$\sigma_{q_n}$ PN Opening-Specific Measured",
    "$\sigma_{q_n}$ IN Bulk Fit",
    "$\sigma_{q_n}$ IN Bulk Measured",
    "$\sigma_{q_n}$ IN Opening-Specific Measured",
    "$\sigma_{\Delta C_p}$ PN Bulk Fit",
    "$\sigma_{\Delta C_p}$ PN Bulk Measured",
    "$\sigma_{\Delta C_p}$ PN Opening-Specific Measured",
    "$\sigma_{\Delta C_p}$ IN Bulk Fit",
    "$\sigma_{\Delta C_p}$ IN Bulk Measured",
    "$\sigma_{\Delta C_p}$ IN Opening-Specific Measured",
]

original_table_label_map = {
    ashrae_baseline_label: ashrae_baseline_label,
    ashrae_q_label: ashrae_q_label,
    ashrae_p_label: ashrae_p_label,
    "Mean-Pressure Baseline": "LES Mean-Pressure Baseline",
    "Mean-Pressure Fit PN": "LES Mean-Pressure PN Fit",
    "$\sigma_{q_n}$ PN Bulk Fit": "LES $\sigma_{q_n}$ PN Bulk Fit",
    "$\sigma_{q_n}$ PN Bulk Measured": "LES $\sigma_{q_n}$ PN Bulk Measured",
    "$\sigma_{q_n}$ PN Opening-Specific Measured": "LES $\sigma_{q_n}$ PN Opening-Specific Measured",
    "$\sigma_{q_n}$ IN Bulk Fit": "LES $\sigma_{q_n}$ IN Bulk Fit",
    "$\sigma_{q_n}$ IN Bulk Measured": "LES $\sigma_{q_n}$ IN Bulk Measured",
    "$\sigma_{q_n}$ IN Opening-Specific Measured": "LES $\sigma_{q_n}$ IN Opening-Specific Measured",
    "$\sigma_{\Delta C_p}$ PN Bulk Fit": "LES $\sigma_{\Delta C_p}$ PN Bulk Fit",
    "$\sigma_{\Delta C_p}$ PN Bulk Measured": "LES $\sigma_{\Delta C_p}$ PN Bulk Measured",
    "$\sigma_{\Delta C_p}$ PN Opening-Specific Measured": "LES $\sigma_{\Delta C_p}$ PN Opening-Specific Measured",
    "$\sigma_{\Delta C_p}$ IN Bulk Fit": "LES $\sigma_{\Delta C_p}$ IN Bulk Fit",
    "$\sigma_{\Delta C_p}$ IN Bulk Measured": "LES $\sigma_{\Delta C_p}$ IN Bulk Measured",
    "$\sigma_{\Delta C_p}$ IN Opening-Specific Measured": "LES $\sigma_{\Delta C_p}$ IN Opening-Specific Measured",
}

def _add_iqr_error_column(df):
    if "iqr_error" in df.columns:
        return df
    for candidate in ["iqr", "error_iqr"]:
        if candidate in df.columns:
            df = df.copy()
            df["iqr_error"] = df[candidate]
            return df
    df = df.copy()
    df["iqr_error"] = np.nan
    return df

def _add_r_squared_column(df):
    if "r_squared" in df.columns:
        return df
    df = df.copy()
    df["r_squared"] = 1 - df["nrmse"] ** 2
    return df

def _ashrae_model_map():
    return {
        "ASHRAE Pseudo-Steady Baseline": ashrae_baseline_label,
        "ASHRAE Mean-Pressure Baseline": ashrae_baseline_label,
        "ASHRAE Baseline": ashrae_baseline_label,
        "Mean-Pressure Baseline": ashrae_baseline_label,
        ashrae_q_label: ashrae_q_label,
        "ASHRAE $\\sigma_{q_n}$ Fit": ashrae_q_label,
        "ASHRAE $\\sigma_{q_n}$ Bulk Fit": ashrae_q_label,
        "$\\sigma_{q_n}$ ASHRAE-Corrected": ashrae_q_label,
        ashrae_p_label: ashrae_p_label,
        "ASHRAE $\\sigma_{\\Delta C_p}$ Fit": ashrae_p_label,
        "ASHRAE $\\sigma_{\\Delta C_p}$ Bulk Fit": ashrae_p_label,
        "$\\sigma_{\\Delta C_p}$ ASHRAE-Corrected": ashrae_p_label,
    }

def _ashrae_fit_param_lookup(ashrae_fit_params_df):
    if ashrae_fit_params_df.empty:
        return pd.DataFrame(columns=["cd", "sigma"])
    direction_map = {"Flow Entering": "in (+)", "Flow Exiting": "out (-)"}
    fit_params = ashrae_fit_params_df.copy()
    fit_params["table_model"] = fit_params["model_name"].map(_ashrae_model_map()).fillna(fit_params["model_name"])
    fit_params["direction_symbol"] = fit_params["direction"].map(direction_map)
    fit_params = fit_params.dropna(subset=["table_model", "direction_symbol"])
    if fit_params.empty:
        return pd.DataFrame(columns=["cd", "sigma"])
    return fit_params.set_index(["table_model", "direction_symbol"])[["cd", "sigma"]]

def _ashrae_opening_metrics_table(ashrae_error_df, ashrae_fit_params_df):
    if ashrae_error_df.empty:
        return pd.DataFrame(columns=["model_name", "table_model", "opening_group", "direction_symbol", "cd", "sigma", "rmse", "bias", "iqr_error", "nrmse", "r_squared", "n"])

    ashrae_error_df = _add_r_squared_column(_add_iqr_error_column(ashrae_error_df))
    model_map = _ashrae_model_map()
    fit_param_lookup = _ashrae_fit_param_lookup(ashrae_fit_params_df)
    direction_map = {"Flow Entering": "in (+)", "Flow Exiting": "out (-)"}

    opening_ashrae = (
        ashrae_error_df[
            (ashrae_error_df["dataset"] == "windows")
            & (ashrae_error_df["scope"] == "group")
            & (ashrae_error_df["model_name"].isin(model_map))
        ]
        .copy()
    )
    if opening_ashrae.empty:
        return pd.DataFrame(columns=["model_name", "table_model", "opening_group", "direction_symbol", "cd", "sigma", "rmse", "bias", "iqr_error", "nrmse", "r_squared", "n"])

    opening_ashrae["table_model"] = opening_ashrae["model_name"].map(model_map)
    opening_ashrae["model_name"] = opening_ashrae["table_model"]
    opening_ashrae["opening_group"] = "Window"
    opening_ashrae["direction_symbol"] = opening_ashrae["group"].map(direction_map)
    opening_ashrae["cd"] = np.nan
    opening_ashrae["sigma"] = np.nan
    if not fit_param_lookup.empty:
        lookup_index = opening_ashrae.set_index(["table_model", "direction_symbol"]).index
        opening_ashrae["cd"] = [fit_param_lookup["cd"].get(key, np.nan) for key in lookup_index]
        opening_ashrae["sigma"] = [fit_param_lookup["sigma"].get(key, np.nan) for key in lookup_index]
    opening_ashrae["opening_group"] = pd.Categorical(opening_ashrae["opening_group"], categories=["Window", "Skylight"], ordered=True)
    opening_ashrae["direction_symbol"] = pd.Categorical(opening_ashrae["direction_symbol"], categories=["in (+)", "out (-)"], ordered=True)
    opening_ashrae["model_order"] = pd.Categorical(opening_ashrae["table_model"], categories=original_table_model_order, ordered=True)
    return opening_ashrae[["model_name", "table_model", "opening_group", "direction_symbol", "model_order", "cd", "sigma", "rmse", "bias", "iqr_error", "nrmse", "r_squared", "n"]]

def _ashrae_room_metrics_table(ashrae_error_df):
    if ashrae_error_df.empty:
        return pd.DataFrame(columns=["model_name", "table_model", "opening_group", "room_type", "rmse", "bias", "iqr_error", "nrmse", "r_squared", "n"])

    ashrae_error_df = _add_r_squared_column(_add_iqr_error_column(ashrae_error_df))
    model_map = _ashrae_model_map()

    room_ashrae = (
        ashrae_error_df[
            (ashrae_error_df["dataset"] == "rooms")
            & (ashrae_error_df["scope"] == "group")
            & (ashrae_error_df["model_name"].isin(model_map))
        ]
        .copy()
    )
    if room_ashrae.empty:
        return pd.DataFrame(columns=["model_name", "table_model", "opening_group", "room_type", "rmse", "bias", "iqr_error", "nrmse", "r_squared", "n"])

    room_ashrae["table_model"] = room_ashrae["model_name"].map(model_map)
    room_ashrae["model_name"] = room_ashrae["table_model"]
    room_ashrae["opening_group"] = "Window"
    room_ashrae["room_type"] = room_ashrae["group"]
    room_ashrae = room_ashrae[room_ashrae["room_type"].isin(room_type_order)].copy()
    room_ashrae["opening_group"] = pd.Categorical(room_ashrae["opening_group"], categories=["Window", "Skylight"], ordered=True)
    room_ashrae["room_type"] = pd.Categorical(room_ashrae["room_type"], categories=room_type_order, ordered=True)
    room_ashrae["model_order"] = pd.Categorical(room_ashrae["table_model"], categories=original_table_model_order, ordered=True)
    return room_ashrae[["model_name", "table_model", "opening_group", "room_type", "model_order", "rmse", "bias", "iqr_error", "nrmse", "r_squared", "n"]]

error_metrics_tidy = _add_r_squared_column(_add_iqr_error_column(error_metrics_tidy))

opening_metric_values = (
    error_metrics_tidy[
        (error_metrics_tidy["source"].isin(["plot_ventilation_model_fit", "direct_network_i_type"]))
        & (error_metrics_tidy["scope"] == "subgroup")
        & (error_metrics_tidy["model_name"].isin(original_table_model_order))
    ]
    .assign(
        table_model=lambda df: df["model_name"],
        opening_group=lambda df: pd.Categorical(
            df["opening_group"], categories=["Window", "Skylight"], ordered=True
        ),
        direction_symbol=lambda df: pd.Categorical(
            df["direction"].map({"Flow Entering": "in (+)", "Flow Exiting": "out (-)"}),
            categories=["in (+)", "out (-)"],
            ordered=True,
        ),
        model_order=lambda df: pd.Categorical(
            df["model_name"], categories=original_table_model_order, ordered=True
        ),
    )
)

original_table_metrics = (
    opening_metric_values
    .groupby(
        ["model_name", "table_model", "opening_group", "direction_symbol", "model_order"],
        observed=True,
        dropna=False,
        sort=False,
    )
    .agg(
        cd=("cd", "mean"),
        sigma=("sigma", "mean"),
        rmse=("rmse", "mean"),
        bias=("bias", "mean"),
        iqr_error=("iqr_error", "mean"),
        nrmse=("nrmse", "mean"),
        r_squared=("r_squared", "mean"),
        n=("n", "sum"),
    )
    .reset_index()
    .sort_values(["opening_group", "model_order", "direction_symbol"])
    .drop(columns=["model_order"])
)

ashrae_error_metrics_df = pd.read_csv(ashrae_error_metrics_path) if os.path.exists(ashrae_error_metrics_path) else pd.DataFrame()
ashrae_fit_params_df = pd.read_csv(ashrae_fit_params_path) if os.path.exists(ashrae_fit_params_path) else pd.DataFrame()
ashrae_opening_metrics = _ashrae_opening_metrics_table(ashrae_error_metrics_df, ashrae_fit_params_df)
if not ashrae_opening_metrics.empty:
    original_table_metrics = (
        pd.concat([original_table_metrics, ashrae_opening_metrics.drop(columns=["model_order"])], ignore_index=True, sort=False)
        .assign(model_order=lambda df: pd.Categorical(df["table_model"], categories=original_table_model_order, ordered=True))
        .sort_values(["opening_group", "model_order", "direction_symbol"])
        .drop(columns=["model_order"])
        .reset_index(drop=True)
    )

room_metric_values = (
    error_metrics_tidy[
        (error_metrics_tidy["source"].isin(["plot_ventilation_model_fit", "direct_network_i_type"]))
        & (error_metrics_tidy["scope"] == "room_slall")
        & (error_metrics_tidy["room_type"].isin(room_type_order))
        & (error_metrics_tidy["model_name"].isin(original_table_model_order))
    ]
    .assign(
        table_model=lambda df: df["model_name"],
        opening_group=lambda df: pd.Categorical(
            df["opening_group"], categories=["Window", "Skylight"], ordered=True
        ),
        room_type=lambda df: pd.Categorical(
            df["room_type"], categories=room_type_order, ordered=True
        ),
        model_order=lambda df: pd.Categorical(
            df["model_name"], categories=original_table_model_order, ordered=True
        ),
    )
)

original_table_room_type_metrics = (
    room_metric_values
    .groupby(
        ["model_name", "table_model", "opening_group", "room_type", "model_order"],
        observed=True,
        dropna=False,
        sort=False,
    )
    .agg(
        rmse=("rmse", "mean"),
        bias=("bias", "mean"),
        iqr_error=("iqr_error", "mean"),
        nrmse=("nrmse", "mean"),
        r_squared=("r_squared", "mean"),
        n=("n", "sum"),
    )
    .reset_index()
    .sort_values(["opening_group", "room_type", "model_order"])
    .drop(columns=["model_order"])
)

ashrae_room_metrics = _ashrae_room_metrics_table(ashrae_error_metrics_df)
if not ashrae_room_metrics.empty:
    original_table_room_type_metrics = (
        pd.concat([original_table_room_type_metrics, ashrae_room_metrics.drop(columns=["model_order"])], ignore_index=True, sort=False)
        .assign(model_order=lambda df: pd.Categorical(df["table_model"], categories=original_table_model_order, ordered=True))
        .sort_values(["opening_group", "room_type", "model_order"])
        .drop(columns=["model_order"])
        .reset_index(drop=True)
    )

opening_r_squared = (
    original_table_metrics
    .groupby(["table_model", "opening_group"], observed=True)["r_squared"]
    .mean()
    .unstack("opening_group")
    .rename(columns={"Window": "Windows", "Skylight": "Skylights"})
)
room_r_squared = (
    original_table_room_type_metrics
    .groupby(["table_model", "opening_group"], observed=True)["r_squared"]
    .mean()
    .unstack("opening_group")
    .rename(
        columns={
            "Window": "Interiors Without Skylights",
            "Skylight": "Interiors With Skylights",
        }
    )
)
r_squared_df = (
    opening_r_squared
    .join(room_r_squared, how="outer")
    .reindex(original_table_model_order)
    .rename(index=original_table_label_map)
    .rename_axis("Model")
    .reset_index()
)
r_squared_df.columns.name = None
display(r_squared_df)

display(
    original_table_metrics[
        [
            "table_model",
            "opening_group",
            "direction_symbol",
            "cd",
            "sigma",
            "rmse",
            "bias",
            "iqr_error",
            "nrmse",
            "n",
        ]
    ].assign(
        table_model=lambda df: df["table_model"].map(original_table_label_map).fillna(df["table_model"])
    ).rename(
        columns={
            "table_model": "Model",
            "opening_group": "Opening",
            "direction_symbol": "Direction",
            "cd": "C_d",
            "rmse": "Opening RMSE Mean",
            "bias": "Opening Bias Mean",
            "iqr_error": "Opening Error IQR Mean",
            "nrmse": "Opening NRMSE Mean",
            "n": "Opening n",
        }
    )
)

display(
    original_table_room_type_metrics[
        [
            "table_model",
            "opening_group",
            "room_type",
            "rmse",
            "bias",
            "iqr_error",
            "nrmse",
            "n",
        ]
    ].assign(
        table_model=lambda df: df["table_model"].map(original_table_label_map).fillna(df["table_model"])
    ).rename(
        columns={
            "table_model": "Model",
            "opening_group": "Room Group",
            "room_type": "Room Type",
            "rmse": "Room-Type RMSE Mean",
            "bias": "Room-Type Bias Mean",
            "iqr_error": "Room-Type Error IQR Mean",
            "nrmse": "Room-Type NRMSE Mean",
            "n": "Room-Type n",
        }
    )
)


In [ ]:
r_squared_df

In [ ]:
from matplotlib.lines import Line2D

# Plotted models and their labels, top-to-bottom, matching the paper table (Fit models omitted).
model_tick_labels = {
    ashrae_baseline_label: "ASHRAE Mean-Pressure Baseline",
    "Mean-Pressure Baseline": "LES Mean-Pressure Baseline",
    r"$\sigma_{q_n}$ IN Opening-Specific Measured": r"LES $\sigma_{q_n}$ IN Opening-Specific",
    r"$\sigma_{q_n}$ IN Bulk Measured": r"LES $\sigma_{q_n}$ IN Bulk",
    r"$\sigma_{q_n}$ PN Bulk Measured": r"LES $\sigma_{q_n}$ PN Bulk",
    r"$\sigma_{\Delta C_p}$ IN Opening-Specific Measured": r"LES $\sigma_{\Delta C_p}$ IN Opening-Specific",
    r"$\sigma_{\Delta C_p}$ IN Bulk Measured": r"LES $\sigma_{\Delta C_p}$ IN Bulk",
    r"$\sigma_{\Delta C_p}$ PN Bulk Measured": r"LES $\sigma_{\Delta C_p}$ PN Bulk",
}
# Matplotlib draws y=0 at the bottom, so reverse the desired visual top-to-bottom order.
model_order = list(model_tick_labels)[::-1]
model_group_sizes = [1, 1, 3, 3]
model_group_boundaries = np.cumsum(model_group_sizes[::-1])[:-1] - 0.5
metric_rows = [("rmse", "RMSE"), ("bias", "Bias")]
opening_cols = [("Window", "Windows"), ("Skylight", "Skylights")]
direction_palette = {"in (+)": "red", "out (-)": "blue"}


def _plot_point(ax, y_pos, value, color):
    if pd.isna(value):
        return
    ax.scatter(
        value,
        y_pos,
        color=color,
        edgecolor="black",
        linewidth=0.6,
        s=42,
        zorder=3,
    )


def _plot_connector(ax, y_pos, values):
    values = [value for value in values if pd.notna(value)]
    if len(values) < 2:
        return
    ax.plot(
        values,
        [y_pos] * len(values),
        color="0.35",
        linewidth=1.2,
        alpha=0.25,
        zorder=2,
    )


def _style_model_labels(ax, show_labels):
    ax.set_yticks(range(len(model_order)))
    if not show_labels:
        ax.tick_params(axis="y", labelleft=False)
        return

    label_left = -0.92
    tick_labels = ax.set_yticklabels(
        [model_tick_labels[model] for model in model_order],
        fontsize=16,
        ha="left",
        va="center",
    )
    for label in tick_labels:
        label.set_x(label_left)
        label.set_clip_on(False)
        label.set_bbox({"facecolor": "white", "edgecolor": "none", "pad": 1})
    for y_pos in range(len(model_order)):
        ax.hlines(
            y_pos,
            xmin=label_left,
            xmax=0,
            transform=ax.get_yaxis_transform(),
            color="0.78",
            linewidth=0.7,
            linestyle="--",
            clip_on=False,
            zorder=0,
        )
    ax.tick_params(axis="y", length=0, pad=0)


fig_opening, axs_opening = plt.subplots(
    len(opening_cols),
    len(metric_rows),
    figsize=(11.5, 8.0),
    dpi=160,
    sharex=False,
    sharey=True,
)

for row_idx, (opening, opening_label) in enumerate(opening_cols):
    for col_idx, (metric_col, metric_label) in enumerate(metric_rows):
        ax = axs_opening[row_idx, col_idx]
        opening_data = original_table_metrics[original_table_metrics["opening_group"] == opening]

        for model_idx, model in enumerate(model_order):
            point_values = []
            for direction in ["in (+)", "out (-)"]:
                row = opening_data[
                    (opening_data["table_model"] == model)
                    & (opening_data["direction_symbol"] == direction)
                ]
                if row.empty:
                    continue
                value = row.iloc[0][metric_col]
                point_values.append(value)
                _plot_point(
                    ax,
                    model_idx,
                    value,
                    direction_palette[direction],
                )
            _plot_connector(ax, model_idx, point_values)

        ax.axvline(0, color="k", linewidth=0.8, alpha=0.7)
        for y_boundary in model_group_boundaries:
            ax.axhline(y_boundary, color="k", linewidth=0.9, alpha=0.7)
        ax.grid(axis="x", alpha=0.25)
        ax.grid(axis="y", alpha=0.35, linestyle="--")
        _style_model_labels(ax, show_labels=col_idx == 0)
        ax.tick_params(axis="x", labelsize=16)
        ax.tick_params(axis="y", labelsize=16)
        ax.set_ylim(-0.5, len(model_order) - 0.5)
        if row_idx == 0:
            ax.set_title(metric_label, fontsize=18)
        if col_idx == 0:
            ax.set_ylabel(opening_label, fontsize=16)

flow_handles = [
    Line2D([0], [0], color=direction_palette[direction], marker="o", linestyle="None", markersize=9, label=f"{direction}")
    for direction in ["in (+)", "out (-)"]
]
fig_opening.legend(
    handles=flow_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=2,
    fontsize=16,
    frameon=False,
)
fig_opening.tight_layout(rect=[0, 0.06, 1, 1])

room_types = room_type_order
room_palette = dict(zip(room_types, sns.color_palette("colorblind", n_colors=len(room_types))))
slall_cols = [("Window", "Interiors Without Skylights"), ("Skylight", "Interiors With Skylights")]

if original_table_room_type_metrics.empty:
    raise ValueError("No room-type metrics found for the room-level RMSE/Bias plot.")

fig_room, axs_room = plt.subplots(
    len(slall_cols),
    len(metric_rows),
    figsize=(11.5, 8.0),
    dpi=160,
    sharex=False,
    sharey=True,
)

for row_idx, (opening_group, title) in enumerate(slall_cols):
    for col_idx, (metric_col, metric_label) in enumerate(metric_rows):
        ax = axs_room[row_idx, col_idx]
        slall_data = original_table_room_type_metrics[original_table_room_type_metrics["opening_group"] == opening_group]

        for model_idx, model in enumerate(model_order):
            point_values = []
            for room_type in room_types:
                row = slall_data[
                    (slall_data["table_model"] == model)
                    & (slall_data["room_type"] == room_type)
                ]
                if row.empty:
                    continue
                value = row.iloc[0][metric_col]
                point_values.append(value)
                _plot_point(
                    ax,
                    model_idx,
                    value,
                    room_palette[room_type],
                )
            _plot_connector(ax, model_idx, point_values)

        ax.axvline(0, color="k", linewidth=0.8, alpha=0.7)
        for y_boundary in model_group_boundaries:
            ax.axhline(y_boundary, color="k", linewidth=0.9, alpha=0.7)
        ax.grid(axis="x", alpha=0.25)
        ax.grid(axis="y", alpha=0.35, linestyle="--")
        _style_model_labels(ax, show_labels=col_idx == 0)
        ax.tick_params(axis="x", labelsize=16)
        ax.tick_params(axis="y", labelsize=16)
        ax.set_ylim(-0.5, len(model_order) - 0.5)
        if row_idx == 0:
            ax.set_title(metric_label, fontsize=18)
        if col_idx == 0:
            ax.set_ylabel(title, fontsize=16)

room_handles = [
    Line2D(
        [0], [0],
        color=room_palette[room_type],
        marker="o",
        linestyle="None",
        markersize=9,
        label=room_type_display_labels.get(room_type, room_type.title()),
    )
    for room_type in room_types
]
fig_room.legend(
    handles=room_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=len(room_types),
    fontsize=16,
    frameon=False,
)
fig_room.tight_layout(rect=[0, 0.06, 1, 1])
